### data\sec_filings_core_pipeline.py

In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import csv
import os
import shutil
import threading
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from pathlib import Path
from typing import Any

import requests


# ============================================================
# PATHS / CONFIG
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

DEFAULT_INPUT_MANIFEST = (
    dataPath / "sec_edgar" / "processed" / "manifests" / "filings_manifest_2000_2024_merged.csv"
)

DEFAULT_FILTERED_MANIFEST = (
    dataPath / "sec_edgar" / "processed" / "manifests" / "filings_manifest_core_companyscope_2000_2024.csv"
)

DEFAULT_CIK_MAP = (
    dataPath / "sec_edgar" / "processed" / "cleaned" / "cik_ticker_map_cleaned.csv"
)

DEFAULT_ISSUER_TICKERS = (
    dataPath / "sec_edgar" / "processed" / "cleaned" / "issuer_master_onlyTickers.csv"
)

RAW_FILINGS_DIR = dataPath / "sec_edgar" / "raw" / "filings_txt"
LOG_DIR = dataPath / "sec_edgar" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_LOG = LOG_DIR / "core_filings_companyscope_download_log.csv"
SEC_USER_AGENT = os.getenv(
    "SEC_USER_AGENT",
    "IbrahimHussain ibrahimbeaconarion@gmail.com"
).strip()

CORE_FORMS = {"10-K", "10-Q", "8-K", "DEF 14A"}

REQUEST_TIMEOUT = (15, 120)
TRANSIENT_RETRY_STATUSES = {429, 500, 502, 503, 504}
PROGRESS_EVERY = 1000

_thread_local = threading.local()


# ============================================================
# HELPERS
# ============================================================

def require_user_agent() -> None:
    if not SEC_USER_AGENT:
        raise SystemExit(
            "SEC_USER_AGENT is not set.\n"
            'Example:\nexport SEC_USER_AGENT="YourName your_email@example.com"'
        )


def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if n < 1024.0 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024.0
    return f"{n:.2f} B"


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def free_bytes(path: Path) -> int:
    usage = shutil.disk_usage(path)
    return int(usage.free)


def form_is_core(form_type: str) -> bool:
    return form_type.strip().upper() in CORE_FORMS


def year_in_range(date_filed: str, start_year: int, end_year: int) -> bool:
    if not date_filed or len(date_filed) < 4:
        return False
    try:
        y = int(date_filed[:4])
        return start_year <= y <= end_year
    except Exception:
        return False


def get_session() -> requests.Session:
    sess = getattr(_thread_local, "session", None)
    if sess is None:
        sess = requests.Session()
        sess.headers.update({
            "User-Agent": SEC_USER_AGENT,
            "Accept-Encoding": "gzip, deflate",
        })
        _thread_local.session = sess
    return sess


def log_csv_header_if_needed(path: Path, header: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(header)


def load_permanent_failures(log_path: Path) -> set[str]:
    permanent: set[str] = set()
    if not log_path.exists():
        return permanent

    with log_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            status = (row.get("status") or "").strip()
            url = (row.get("filing_url") or "").strip()
            if status in {"404", "410"} and url:
                permanent.add(url)
    return permanent


def append_log_row(log_path: Path, row: dict[str, Any]) -> None:
    log_csv_header_if_needed(
        log_path,
        ["timestamp", "filing_url", "output_path", "status", "bytes_written", "error_message"],
    )
    with log_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            row.get("timestamp", ""),
            row.get("filing_url", ""),
            row.get("output_path", ""),
            row.get("status", ""),
            row.get("bytes_written", ""),
            row.get("error_message", ""),
        ])


def safe_mkdir_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def save_response_to_file(resp: requests.Response, dest: Path) -> int:
    safe_mkdir_parent(dest)
    tmp = dest.with_suffix(dest.suffix + ".part")

    written = 0
    with tmp.open("wb") as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                written += len(chunk)

    tmp.replace(dest)
    return written


def normalize_cik(value: Any) -> str:
    s = str(value).strip()
    if not s:
        return ""
    # manifest CIKs are usually unpadded numerics
    s = s.lstrip("0")
    return s if s else "0"


def detect_cik_column(fieldnames: list[str]) -> str:
    lowered = {c.lower(): c for c in fieldnames}
    candidates = [
        "cik",
        "padded_cik",
        "sec_cik",
        "issuer_cik",
    ]
    for cand in candidates:
        if cand in lowered:
            return lowered[cand]
    raise RuntimeError(f"Could not find a CIK column in: {fieldnames}")


def load_cik_union(csv_paths: list[Path]) -> set[str]:
    """
    Load union of CIKs into RAM.
    This is tiny enough to keep in memory.
    """
    cik_set: set[str] = set()

    for path in csv_paths:
        if not path.exists():
            raise FileNotFoundError(f"Required CIK source file not found: {path}")

        with path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames or []
            if not fieldnames:
                raise RuntimeError(f"CSV has no header: {path}")

            cik_col = detect_cik_column(fieldnames)

            loaded = 0
            for row in reader:
                cik = normalize_cik(row.get(cik_col, ""))
                if cik:
                    cik_set.add(cik)
                    loaded += 1

            print(
                f"[cik-union] loaded {loaded:,} rows from {path} | cumulative unique CIKs={len(cik_set):,}",
                flush=True,
            )

    return cik_set


# ============================================================
# STAGE 1 — FILTER MANIFEST
# ============================================================

def filter_manifest_by_cik_and_form(
    input_manifest: Path,
    output_manifest: Path,
    cik_union: set[str],
    start_year: int,
    end_year: int,
    overwrite: bool,
) -> None:
    if output_manifest.exists() and not overwrite:
        print(f"[filter] Output already exists: {output_manifest}")
        print("[filter] Use --overwrite to rebuild it.")
        return

    output_manifest.parent.mkdir(parents=True, exist_ok=True)

    total_in = 0
    kept_form_year = 0
    kept_final = 0
    dropped_non_core = 0
    dropped_year = 0
    dropped_cik = 0

    start = time.time()

    with input_manifest.open("r", newline="", encoding="utf-8") as in_f, \
         output_manifest.open("w", newline="", encoding="utf-8") as out_f:
        reader = csv.DictReader(in_f)
        fieldnames = reader.fieldnames
        if not fieldnames:
            raise RuntimeError(f"Manifest has no header: {input_manifest}")

        required = {"cik", "form_type", "date_filed", "filing_url", "output_path"}
        missing = required - set(fieldnames)
        if missing:
            raise RuntimeError(f"Manifest missing required columns: {sorted(missing)}")

        writer = csv.DictWriter(out_f, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            total_in += 1

            form_type = row["form_type"]
            date_filed = row["date_filed"]
            cik = normalize_cik(row["cik"])

            if not form_is_core(form_type):
                dropped_non_core += 1
                continue

            if not year_in_range(date_filed, start_year, end_year):
                dropped_year += 1
                continue

            kept_form_year += 1

            if cik not in cik_union:
                dropped_cik += 1
                continue

            writer.writerow(row)
            kept_final += 1

            if total_in % 500_000 == 0:
                elapsed = time.time() - start
                rate = total_in / max(elapsed, 1e-9)
                print(
                    f"[filter] scanned={total_in:,} kept={kept_final:,} "
                    f"| kept_form_year={kept_form_year:,} dropped_cik={dropped_cik:,} "
                    f"| rate={rate:,.0f} rows/s | elapsed={fmt_elapsed(elapsed)}",
                    flush=True,
                )

    elapsed = time.time() - start
    print(
        f"[filter] done. scanned={total_in:,} kept={kept_final:,} "
        f"| dropped_non_core={dropped_non_core:,} dropped_year={dropped_year:,} dropped_cik={dropped_cik:,} "
        f"| elapsed={fmt_elapsed(elapsed)} | output={output_manifest}",
        flush=True,
    )


# ============================================================
# STAGE 2 — DOWNLOAD
# ============================================================

def pre_scan_manifest(
    manifest_path: Path,
    permanent_failures: set[str],
) -> dict[str, Any]:
    total_rows = 0
    skip_existing = 0
    skip_permanent = 0
    pending = 0
    sample_sizes: list[int] = []

    with manifest_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total_rows += 1
            filing_url = row["filing_url"]
            output_path = Path(row["output_path"])

            if filing_url in permanent_failures:
                skip_permanent += 1
                continue

            if output_path.exists():
                skip_existing += 1
                try:
                    sz = output_path.stat().st_size
                    if sz > 0 and len(sample_sizes) < 5000:
                        sample_sizes.append(sz)
                except Exception:
                    pass
                continue

            pending += 1

            if total_rows % 250_000 == 0:
                print(
                    f"[prescan] scanned={total_rows:,} pending={pending:,} "
                    f"skip_existing={skip_existing:,} skip_permanent={skip_permanent:,}",
                    flush=True,
                )

    avg_existing_size = (sum(sample_sizes) / len(sample_sizes)) if sample_sizes else None
    estimated_required_bytes = int(avg_existing_size * pending) if avg_existing_size else None

    return {
        "total_rows": total_rows,
        "skip_existing": skip_existing,
        "skip_permanent": skip_permanent,
        "pending": pending,
        "avg_existing_size": avg_existing_size,
        "estimated_required_bytes": estimated_required_bytes,
    }


def download_one(row: dict[str, str], max_retries: int) -> dict[str, Any]:
    filing_url = row["filing_url"]
    output_path = Path(row["output_path"])

    if output_path.exists():
        return {
            "status": "exists",
            "filing_url": filing_url,
            "output_path": str(output_path),
            "bytes_written": 0,
            "error_message": "",
        }

    sess = get_session()
    last_error = ""

    for attempt in range(max_retries + 1):
        try:
            resp = sess.get(filing_url, timeout=REQUEST_TIMEOUT, stream=True)

            if resp.status_code in {404, 410}:
                return {
                    "status": str(resp.status_code),
                    "filing_url": filing_url,
                    "output_path": str(output_path),
                    "bytes_written": 0,
                    "error_message": f"{resp.status_code} Client Error",
                }

            if resp.status_code in TRANSIENT_RETRY_STATUSES:
                last_error = f"HTTP {resp.status_code}"
                if attempt < max_retries:
                    wait_time = min(8, 2 ** attempt)
                    time.sleep(wait_time)
                    continue
                return {
                    "status": "failed",
                    "filing_url": filing_url,
                    "output_path": str(output_path),
                    "bytes_written": 0,
                    "error_message": last_error,
                }

            resp.raise_for_status()
            written = save_response_to_file(resp, output_path)

            return {
                "status": "downloaded",
                "filing_url": filing_url,
                "output_path": str(output_path),
                "bytes_written": written,
                "error_message": "",
            }

        except requests.HTTPError as exc:
            code = getattr(exc.response, "status_code", None)
            if code in {404, 410}:
                return {
                    "status": str(code),
                    "filing_url": filing_url,
                    "output_path": str(output_path),
                    "bytes_written": 0,
                    "error_message": str(exc),
                }

            last_error = str(exc)
            if code in TRANSIENT_RETRY_STATUSES and attempt < max_retries:
                wait_time = min(8, 2 ** attempt)
                time.sleep(wait_time)
                continue
            break

        except (requests.ConnectionError, requests.Timeout, requests.RequestException) as exc:
            last_error = str(exc)
            if attempt < max_retries:
                wait_time = min(8, 2 ** attempt)
                time.sleep(wait_time)
                continue
            break

        except Exception as exc:
            last_error = str(exc)
            break

    return {
        "status": "failed",
        "filing_url": filing_url,
        "output_path": str(output_path),
        "bytes_written": 0,
        "error_message": last_error,
    }


def download_manifest(
    manifest_path: Path,
    workers: int,
    max_pending_futures: int,
    min_free_gb: float,
    max_retries: int,
    force: bool,
) -> None:
    min_free_bytes = int(min_free_gb * (1024 ** 3))

    current_free = free_bytes(RAW_FILINGS_DIR)
    print(f"[download] current free disk: {human_bytes(current_free)}", flush=True)
    if current_free < min_free_bytes:
        raise SystemExit(
            f"Refusing to start: free disk space {human_bytes(current_free)} "
            f"is below the required minimum of {human_bytes(min_free_bytes)}"
        )

    permanent_failures = load_permanent_failures(DOWNLOAD_LOG)
    print(f"[download] loaded {len(permanent_failures):,} permanent 404/410 failures from log", flush=True)

    pre = pre_scan_manifest(manifest_path, permanent_failures)
    print(
        f"[download] manifest rows={pre['total_rows']:,} "
        f"pending={pre['pending']:,} "
        f"skip_existing={pre['skip_existing']:,} "
        f"skip_permanent={pre['skip_permanent']:,}",
        flush=True,
    )

    est = pre["estimated_required_bytes"]
    if est is not None:
        print(
            f"[download] avg existing file size ≈ {human_bytes(pre['avg_existing_size'])} "
            f"| estimated bytes needed for pending ≈ {human_bytes(est)}",
            flush=True,
        )
        safe_available = max(0, current_free - min_free_bytes)
        if est > safe_available and not force:
            raise SystemExit(
                f"Estimated required space {human_bytes(est)} exceeds safe available space "
                f"{human_bytes(safe_available)}.\n"
                f"Free disk={human_bytes(current_free)}, min reserve={human_bytes(min_free_bytes)}.\n"
                f"Use --force to proceed anyway, or free space first."
            )

    start = time.time()
    submitted = 0
    completed = 0
    downloaded = 0
    skipped_existing = 0
    skipped_permanent = 0
    failed = 0
    downloaded_bytes = 0

    pending_futures: dict[Any, dict[str, str]] = {}

    def handle_result(result: dict[str, Any]) -> None:
        nonlocal completed, downloaded, skipped_existing, skipped_permanent, failed, downloaded_bytes

        completed += 1
        status = result["status"]
        if status == "downloaded":
            downloaded += 1
            downloaded_bytes += int(result["bytes_written"] or 0)
            append_log_row(DOWNLOAD_LOG, {
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                **result,
            })
        elif status == "exists":
            skipped_existing += 1
        elif status in {"404", "410"}:
            skipped_permanent += 1
            append_log_row(DOWNLOAD_LOG, {
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                **result,
            })
        else:
            failed += 1
            append_log_row(DOWNLOAD_LOG, {
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                **result,
            })

        if completed % PROGRESS_EVERY == 0:
            elapsed = time.time() - start
            rate = completed / max(elapsed, 1e-9)
            print(
                f"[download] completed={completed:,} submitted={submitted:,} "
                f"downloaded={downloaded:,} exists={skipped_existing:,} "
                f"perm_skip={skipped_permanent:,} failed={failed:,} "
                f"bytes={human_bytes(downloaded_bytes)} | rate={rate:,.1f} items/s "
                f"| elapsed={fmt_elapsed(elapsed)}",
                flush=True,
            )

    try:
        with manifest_path.open("r", newline="", encoding="utf-8") as f, \
             ThreadPoolExecutor(max_workers=workers) as executor:
            reader = csv.DictReader(f)

            for row in reader:
                filing_url = row["filing_url"]
                output_path = Path(row["output_path"])

                if filing_url in permanent_failures:
                    skipped_permanent += 1
                    continue

                if output_path.exists():
                    skipped_existing += 1
                    continue

                if submitted % 500 == 0:
                    current_free = free_bytes(RAW_FILINGS_DIR)
                    if current_free < min_free_bytes:
                        print(
                            f"[download] stopping due to low disk space. "
                            f"current_free={human_bytes(current_free)} < reserve={human_bytes(min_free_bytes)}",
                            flush=True,
                        )
                        break

                while len(pending_futures) >= max_pending_futures:
                    done, _ = wait(pending_futures.keys(), return_when=FIRST_COMPLETED)
                    for fut in done:
                        pending_futures.pop(fut, None)
                        result = fut.result()
                        handle_result(result)

                fut = executor.submit(download_one, row, max_retries)
                pending_futures[fut] = row
                submitted += 1

            while pending_futures:
                done, _ = wait(pending_futures.keys(), return_when=FIRST_COMPLETED)
                for fut in done:
                    pending_futures.pop(fut, None)
                    result = fut.result()
                    handle_result(result)

    except KeyboardInterrupt:
        print(
            "\n[download] interrupted. Already-downloaded files are preserved.\n"
            "[download] Re-run the same command to resume.",
            flush=True,
        )
        raise

    elapsed = time.time() - start
    print(
        f"[download] done. submitted={submitted:,} completed={completed:,} "
        f"downloaded={downloaded:,} exists={skipped_existing:,} "
        f"perm_skip={skipped_permanent:,} failed={failed:,} "
        f"bytes={human_bytes(downloaded_bytes)} | elapsed={fmt_elapsed(elapsed)}",
        flush=True,
    )


# ============================================================
# CLI
# ============================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Filter giant SEC filings manifest by company CIK union + selected forms, then download with resume support."
    )
    sub = parser.add_subparsers(dest="command", required=True)

    p_filter = sub.add_parser("filter-manifest", help="Stream-filter the giant manifest down to unioned-company core forms.")
    p_filter.add_argument("--input", type=str, default=str(DEFAULT_INPUT_MANIFEST))
    p_filter.add_argument("--output", type=str, default=str(DEFAULT_FILTERED_MANIFEST))
    p_filter.add_argument("--cik-map", type=str, default=str(DEFAULT_CIK_MAP))
    p_filter.add_argument("--issuer-tickers", type=str, default=str(DEFAULT_ISSUER_TICKERS))
    p_filter.add_argument("--start-year", type=int, default=2000)
    p_filter.add_argument("--end-year", type=int, default=2024)
    p_filter.add_argument("--overwrite", action="store_true")

    p_download = sub.add_parser("download", help="Download filtered company-scope core filings in parallel.")
    p_download.add_argument("--manifest", type=str, default=str(DEFAULT_FILTERED_MANIFEST))
    p_download.add_argument("--workers", type=int, default=8)
    p_download.add_argument("--max-pending-futures", type=int, default=128)
    p_download.add_argument("--min-free-gb", type=float, default=12.0)
    p_download.add_argument("--max-retries", type=int, default=2)
    p_download.add_argument("--force", action="store_true")

    p_all = sub.add_parser("all", help="Filter by CIK union and forms, then download.")
    p_all.add_argument("--input", type=str, default=str(DEFAULT_INPUT_MANIFEST))
    p_all.add_argument("--output", type=str, default=str(DEFAULT_FILTERED_MANIFEST))
    p_all.add_argument("--cik-map", type=str, default=str(DEFAULT_CIK_MAP))
    p_all.add_argument("--issuer-tickers", type=str, default=str(DEFAULT_ISSUER_TICKERS))
    p_all.add_argument("--start-year", type=int, default=2000)
    p_all.add_argument("--end-year", type=int, default=2024)
    p_all.add_argument("--overwrite", action="store_true")
    p_all.add_argument("--workers", type=int, default=8)
    p_all.add_argument("--max-pending-futures", type=int, default=128)
    p_all.add_argument("--min-free-gb", type=float, default=12.0)
    p_all.add_argument("--max-retries", type=int, default=2)
    p_all.add_argument("--force", action="store_true")

    return parser.parse_args()


def main() -> None:
    require_user_agent()
    args = parse_args()

    RAW_FILINGS_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    if args.command == "filter-manifest":
        cik_union = load_cik_union([Path(args.cik_map), Path(args.issuer_tickers)])
        print(f"[cik-union] final unique CIKs={len(cik_union):,}", flush=True)

        filter_manifest_by_cik_and_form(
            input_manifest=Path(args.input),
            output_manifest=Path(args.output),
            cik_union=cik_union,
            start_year=args.start_year,
            end_year=args.end_year,
            overwrite=args.overwrite,
        )

    elif args.command == "download":
        download_manifest(
            manifest_path=Path(args.manifest),
            workers=args.workers,
            max_pending_futures=args.max_pending_futures,
            min_free_gb=args.min_free_gb,
            max_retries=args.max_retries,
            force=args.force,
        )

    elif args.command == "all":
        cik_union = load_cik_union([Path(args.cik_map), Path(args.issuer_tickers)])
        print(f"[cik-union] final unique CIKs={len(cik_union):,}", flush=True)

        out_manifest = Path(args.output)
        filter_manifest_by_cik_and_form(
            input_manifest=Path(args.input),
            output_manifest=out_manifest,
            cik_union=cik_union,
            start_year=args.start_year,
            end_year=args.end_year,
            overwrite=args.overwrite,
        )

        download_manifest(
            manifest_path=out_manifest,
            workers=args.workers,
            max_pending_futures=args.max_pending_futures,
            min_free_gb=args.min_free_gb,
            max_retries=args.max_retries,
            force=args.force,
        )

    else:
        raise SystemExit(f"Unknown command: {args.command}")


if __name__ == "__main__":
    main()

# Filter only:
# nice -n -20 python -u data/sec_filings_core_pipeline.py filter-manifest \
#   --input data/sec_edgar/processed/manifests/filings_manifest_2000_2024_merged.csv \
#   --output data/sec_edgar/processed/manifests/filings_manifest_core_companyscope_2000_2024.csv \
#   --cik-map data/sec_edgar/processed/cleaned/cik_ticker_map_cleaned.csv \
#   --issuer-tickers data/sec_edgar/processed/cleaned/issuer_master_onlyTickers.csv \
#   --start-year 2000 \
#   --end-year 2024 \
#   --overwrite
#Download only:
# nice -n -20 python -u data/sec_filings_core_pipeline.py download \
#   --manifest data/sec_edgar/processed/manifests/filings_manifest_core_companyscope_2000_2024.csv \
#   --workers 8 \
#   --max-pending-futures 256 \
#   --min-free-gb 12\
#Or both:
# nice -n -20 python -u data/sec_filings_core_pipeline.py all \
#   --input data/sec_edgar/processed/manifests/filings_manifest_2000_2024_merged.csv \
#   --output data/sec_edgar/processed/manifests/filings_manifest_core_companyscope_2000_2024.csv \
#   --cik-map data/sec_edgar/processed/cleaned/cik_ticker_map_cleaned.csv \
#   --issuer-tickers data/sec_edgar/processed/cleaned/issuer_master_onlyTickers.csv \
#   --start-year 2000 \
#   --end-year 2024 \
#   --overwrite \
#   --workers 8 \
#   --max-pending-futures 256 \
#   --min-free-gb 12


### data\sec_filings_text_pipeline.py

In [ ]:
#!/usr/bin/env python3
"""
sec_filings_text_pipeline.py

Linux-targeted end-to-end text engineering pipeline for a downloaded SEC filings corpus.
Designed for storage-constrained, partially downloaded corpora and HDD-backed storage.

MODULES
A. inventory   -> Scan downloaded filings on disk and build raw inventory
B. clean       -> Parse metadata + extract/clean primary text + build full filtered corpus
C. sections    -> Extract section-level text
D. quality     -> Coverage + quality reports
E. datasets    -> Build full/balanced modeling manifests
F. finbert     -> Build FinBERT-ready chunk datasets

Run modules selectively:
    python data/sec_filings_text_pipeline.py inventory
    python data/sec_filings_text_pipeline.py clean --resume
    python data/sec_filings_text_pipeline.py sections --resume
    python data/sec_filings_text_pipeline.py quality
    python data/sec_filings_text_pipeline.py datasets
    python data/sec_filings_text_pipeline.py finbert
    python data/sec_filings_text_pipeline.py all --resume

Key design decisions:
- Minimal repeated I/O on HDD: parse each raw filing once in module B.
- Large text outputs are partitioned and gzip-compressed JSONL.
- Resume-friendly: completed parts are marked and never reprocessed.
- Company-universe filtering uses union of cleaned SEC universe CSVs.
- Balanced dataset is manifest-based (no text duplication).
"""

from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import heapq
import html
import io
import json
import math
import os
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Iterable

try:
    import orjson  # type: ignore
    HAS_ORJSON = True
except Exception:
    orjson = None
    HAS_ORJSON = False


# ============================================================
# GLOBAL PATHS / CONFIG
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

RAW_FILINGS_DIR = dataPath / "sec_edgar" / "raw" / "filings_txt"
CLEANED_DIR = dataPath / "sec_edgar" / "processed" / "cleaned"

DEFAULT_CIK_MAP = CLEANED_DIR / "cik_ticker_map_cleaned.csv"
DEFAULT_ISSUER_TICKERS = CLEANED_DIR / "issuer_master_onlyTickers.csv"

OUT_ROOT = dataPath / "sec_edgar" / "processed" / "filings_text"
PARTS_DIR = OUT_ROOT / "parts"
TMP_DIR = OUT_ROOT / "_tmp"

INVENTORY_CSV = OUT_ROOT / "filings_downloaded_inventory.csv"
FULL_DOCS_MANIFEST = OUT_ROOT / "filings_filtered_full.csv"
FULL_SECTIONS_MANIFEST = OUT_ROOT / "filings_sections_full.csv"

YEAR_COVERAGE_CSV = OUT_ROOT / "filings_year_coverage.csv"
YEAR_FORM_COVERAGE_CSV = OUT_ROOT / "filings_year_form_coverage.csv"
YEAR_COMPANY_COVERAGE_CSV = OUT_ROOT / "filings_year_company_coverage.csv"
SECTION_COVERAGE_CSV = OUT_ROOT / "filings_section_coverage.csv"
QUALITY_SUMMARY_JSON = OUT_ROOT / "filings_quality_summary.json"

BALANCED_DOCS_MANIFEST = OUT_ROOT / "filings_filtered_balanced.csv"
BALANCED_SECTIONS_MANIFEST = OUT_ROOT / "filings_sections_balanced.csv"
BALANCE_SUMMARY_JSON = OUT_ROOT / "filings_balance_summary.json"

FINBERT_CHUNKS_ALL_CSV = OUT_ROOT / "filings_finbert_chunks_all.csv"
FINBERT_CHUNKS_BALANCED_CSV = OUT_ROOT / "filings_finbert_chunks_balanced.csv"
FINBERT_SUMMARY_JSON = OUT_ROOT / "filings_finbert_summary.json"

LOG_DIR = dataPath / "sec_edgar" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
ERRORS_CSV = OUT_ROOT / "filings_text_errors.csv"

ALLOWED_FORMS = {"10-K", "10-Q", "8-K", "DEF 14A"}
ALLOWED_FORM_FOLDERS = {"10-K", "10-Q", "8-K", "DEF_14A"}

COMPRESSLEVEL = 1

INVENTORY_HEADER = [
    "rel_path",
    "abs_path",
    "year",
    "form_folder",
    "file_size_bytes",
    "file_name",
]

DOC_META_HEADER = [
    "doc_id",
    "rel_path",
    "part_id",
    "year",
    "form_type",
    "folder_form",
    "cik",
    "entity_name",
    "filing_date",
    "period_of_report",
    "acceptance_datetime",
    "accession",
    "primary_doc_type",
    "primary_doc_filename",
    "is_company_in_universe",
    "clean_text_chars",
    "clean_text_words",
    "section_hint_count",
    "status",
    "error_message",
]

SECTION_META_HEADER = [
    "section_id",
    "doc_id",
    "part_id",
    "year",
    "form_type",
    "cik",
    "filing_date",
    "accession",
    "section_name",
    "section_rank",
    "source_mode",
    "text_chars",
    "text_words",
]

ERRORS_HEADER = [
    "stage",
    "rel_path",
    "doc_id",
    "error_type",
    "error_message",
]

FORM_ORDER = ["10-K", "10-Q", "8-K", "DEF 14A"]

DEFAULT_FORM_WEIGHTS = {
    "10-K": 0.30,
    "10-Q": 0.30,
    "8-K": 0.30,
    "DEF 14A": 0.10,
}


# ============================================================
# HELPERS
# ============================================================

def json_dumps(obj: Any) -> bytes:
    if HAS_ORJSON:
        return orjson.dumps(obj)
    return json.dumps(obj, ensure_ascii=False).encode("utf-8")


def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for u in units:
        if n < 1024 or u == units[-1]:
            return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} B"


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60
    return f"{hours:.2f}h"


def now_ts() -> float:
    return time.time()


def free_bytes(path: Path) -> int:
    return shutil.disk_usage(path).free


def choose_worker_count(requested: int) -> int:
    if requested > 0:
        return requested
    cpu = os.cpu_count() or 4
    return max(1, min(8, cpu // 2 if cpu >= 4 else 1))


def normalize_cik(value: Any) -> str:
    s = str(value).strip()
    if not s:
        return ""
    s = s.lstrip("0")
    return s if s else "0"


def detect_cik_column(fieldnames: list[str]) -> str:
    lowered = {x.lower(): x for x in fieldnames}
    for name in ("cik", "padded_cik", "sec_cik", "issuer_cik"):
        if name in lowered:
            return lowered[name]
    raise RuntimeError(f"Could not find CIK column in {fieldnames}")


def load_cik_union(csv_paths: list[Path]) -> set[str]:
    cik_set: set[str] = set()
    for path in csv_paths:
        if not path.exists():
            raise FileNotFoundError(path)
        with path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames or []
            cik_col = detect_cik_column(fieldnames)
            n = 0
            for row in reader:
                cik = normalize_cik(row.get(cik_col, ""))
                if cik:
                    cik_set.add(cik)
                    n += 1
            print(f"[cik-union] loaded {n:,} rows from {path} | cumulative unique CIKs={len(cik_set):,}", flush=True)
    return cik_set


def ensure_dirs(overwrite: bool) -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    PARTS_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    if overwrite:
        # only remove stage outputs, not raw data
        targets = [
            INVENTORY_CSV, FULL_DOCS_MANIFEST, FULL_SECTIONS_MANIFEST,
            YEAR_COVERAGE_CSV, YEAR_FORM_COVERAGE_CSV, YEAR_COMPANY_COVERAGE_CSV,
            SECTION_COVERAGE_CSV, QUALITY_SUMMARY_JSON,
            BALANCED_DOCS_MANIFEST, BALANCED_SECTIONS_MANIFEST, BALANCE_SUMMARY_JSON,
            FINBERT_CHUNKS_ALL_CSV, FINBERT_CHUNKS_BALANCED_CSV, FINBERT_SUMMARY_JSON,
            ERRORS_CSV,
        ]
        for t in targets:
            if t.exists():
                t.unlink()
        if PARTS_DIR.exists():
            for p in PARTS_DIR.glob("*"):
                if p.is_file():
                    p.unlink()


def write_csv(path: Path, header: list[str], rows: Iterable[list[Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for row in rows:
            w.writerow(row)
    tmp.replace(path)


def append_csv_row(path: Path, header: list[str], row: list[Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(header)
        w.writerow(row)


def greedy_partition(files: list[Path], shard_count: int) -> list[list[Path]]:
    shard_count = max(1, min(shard_count, len(files)))
    shards = [[] for _ in range(shard_count)]
    heap = [(0, i) for i in range(shard_count)]
    heapq.heapify(heap)
    sized = []
    for p in files:
        try:
            sz = p.stat().st_size
        except Exception:
            sz = 0
        sized.append((sz, p))
    sized.sort(key=lambda x: x[0], reverse=True)
    for sz, p in sized:
        total, idx = heapq.heappop(heap)
        shards[idx].append(p)
        heapq.heappush(heap, (total + sz, idx))
    return [s for s in shards if s]


def load_processed_relpaths(pattern: str, rel_field: str) -> set[str]:
    done: set[str] = set()
    for p in PARTS_DIR.glob(pattern):
        try:
            with p.open("r", newline="", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                if rel_field not in (reader.fieldnames or []):
                    continue
                for row in reader:
                    relp = (row.get(rel_field) or "").replace("\\", "/")
                    if relp:
                        done.add(relp)
        except Exception:
            continue
    return done


def done_marker(base: Path) -> Path:
    return base.with_suffix(base.suffix + ".done")


def normalize_space(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_tags(text: str) -> str:
    # remove scripts/styles first
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", text)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?is)<!--.*?-->", " ", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    return text


def decode_bytes(raw: bytes) -> str:
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return raw.decode(enc)
        except Exception:
            pass
    return raw.decode("utf-8", errors="replace")


def hash_id(*parts: str, n: int = 16) -> str:
    h = hashlib.blake2b(digest_size=16)
    for p in parts:
        h.update(p.encode("utf-8", errors="ignore"))
        h.update(b"\x1f")
    return h.hexdigest()[:n]


SEC_HEADER_PATTERNS = {
    "cik": [
        r"CENTRAL INDEX KEY:\s*([0-9]+)",
        r"CIK:\s*([0-9]+)",
    ],
    "entity_name": [
        r"COMPANY CONFORMED NAME:\s*(.+)",
        r"COMPANY:\s*(.+)",
        r"CONFORMED NAME:\s*(.+)",
    ],
    "form_type": [
        r"CONFORMED SUBMISSION TYPE:\s*([^\n<]+)",
        r"FORM TYPE:\s*([^\n<]+)",
    ],
    "filing_date": [
        r"FILED AS OF DATE:\s*([0-9]{8})",
        r"FILING DATE:\s*([0-9]{8})",
    ],
    "period_of_report": [
        r"CONFORMED PERIOD OF REPORT:\s*([0-9]{8})",
        r"PERIOD OF REPORT:\s*([0-9]{8})",
    ],
    "acceptance_datetime": [
        r"ACCEPTANCE-DATETIME:\s*([0-9]{14})",
    ],
    "accession": [
        r"ACCESSION NUMBER:\s*([0-9\-]+)",
    ],
}


def sec_header_search(text: str, key: str) -> str:
    for pat in SEC_HEADER_PATTERNS[key]:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return ""


def yyyymmdd_to_iso(s: str) -> str:
    if len(s) == 8 and s.isdigit():
        return f"{s[:4]}-{s[4:6]}-{s[6:8]}"
    return s


def folder_form_to_canonical(folder_name: str) -> str:
    return folder_name.replace("_", " ")


def derive_year_form(rel_path: str) -> tuple[str, str]:
    parts = rel_path.replace("\\", "/").split("/")
    # expected: YEAR/FORM/file.txt OR maybe root/YEAR/FORM/file.txt already stripped in rel path
    if len(parts) >= 3:
        year = parts[0]
        folder_form = parts[1]
        return year, folder_form
    return "", ""


def find_document_blocks(text: str) -> list[dict[str, str]]:
    """
    Parse SEC SGML-style <DOCUMENT> blocks if present.
    """
    blocks: list[dict[str, str]] = []
    pattern = re.compile(r"(?is)<DOCUMENT>(.*?)</DOCUMENT>")
    for block in pattern.findall(text):
        doc_type = ""
        filename = ""
        description = ""
        m = re.search(r"(?im)^\s*<TYPE>\s*([^\n\r]+)", block)
        if m:
            doc_type = m.group(1).strip()
        m = re.search(r"(?im)^\s*<FILENAME>\s*([^\n\r]+)", block)
        if m:
            filename = m.group(1).strip()
        m = re.search(r"(?im)^\s*<DESCRIPTION>\s*([^\n\r]+)", block)
        if m:
            description = m.group(1).strip()
        # Body starts after SGML headers; if not clear, keep full block
        body = re.sub(r"(?im)^\s*<(TYPE|SEQUENCE|FILENAME|DESCRIPTION|TEXT)>\s*[^\n\r]*", "", block)
        blocks.append({
            "type": doc_type,
            "filename": filename,
            "description": description,
            "body": body,
        })
    return blocks


def choose_primary_document(full_text: str, declared_form: str) -> tuple[str, str, str]:
    """
    Returns: body, doc_type, doc_filename
    """
    blocks = find_document_blocks(full_text)
    if not blocks:
        return full_text, "", ""

    declared = declared_form.upper().strip()
    normalized_candidates = {
        declared,
        declared.replace("/A", ""),
        declared.replace(" ", ""),
        declared.replace(" ", "_"),
    }

    # best: exact or near-exact type match to declared form
    for b in blocks:
        t = b["type"].upper().strip()
        if t in normalized_candidates or t.replace(" ", "") in normalized_candidates:
            return b["body"], b["type"], b["filename"]

    # next: choose longest non-exhibit-like document
    good = []
    for b in blocks:
        t = b["type"].upper().strip()
        if t.startswith("EX-"):
            continue
        if t in {"GRAPHIC", "XML", "ZIP", "PDF"}:
            continue
        good.append(b)

    if good:
        best = max(good, key=lambda x: len(x["body"]))
        return best["body"], best["type"], best["filename"]

    best = max(blocks, key=lambda x: len(x["body"]))
    return best["body"], best["type"], best["filename"]


def clean_primary_text(body: str) -> str:
    text = html.unescape(body)
    text = strip_tags(text)
    # normalize SGML remnants
    text = re.sub(r"(?im)^\s*<[^>\n]+>\s*$", " ", text)
    text = re.sub(r"[\u00a0\u200b]+", " ", text)
    text = normalize_space(text)
    return text


def count_words(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\b\w+\b", text))


def parse_raw_filing(raw: bytes, rel_path: str, universe_ciks: set[str]) -> tuple[dict[str, Any], str]:
    full_text = decode_bytes(raw)
    header_window = full_text[:250_000]

    year, folder_form = derive_year_form(rel_path)

    cik = normalize_cik(sec_header_search(header_window, "cik"))
    entity_name = sec_header_search(header_window, "entity_name")
    form_type = sec_header_search(header_window, "form_type") or folder_form_to_canonical(folder_form)
    filing_date = yyyymmdd_to_iso(sec_header_search(header_window, "filing_date"))
    period_of_report = yyyymmdd_to_iso(sec_header_search(header_window, "period_of_report"))
    acceptance_datetime = sec_header_search(header_window, "acceptance_datetime")
    accession = sec_header_search(header_window, "accession")

    primary_body, primary_doc_type, primary_doc_filename = choose_primary_document(full_text, form_type)
    clean_text = clean_primary_text(primary_body)

    doc_id = hash_id(rel_path, accession or "", cik or "", form_type or "")
    section_hint_count = len(re.findall(r"(?im)\bitem\s+[0-9]{1,2}[a-z]?(?:\.[0-9]{2})?\b", clean_text))

    meta = {
        "doc_id": doc_id,
        "rel_path": rel_path,
        "year": year,
        "form_type": form_type,
        "folder_form": folder_form,
        "cik": cik,
        "entity_name": entity_name,
        "filing_date": filing_date,
        "period_of_report": period_of_report,
        "acceptance_datetime": acceptance_datetime,
        "accession": accession,
        "primary_doc_type": primary_doc_type,
        "primary_doc_filename": primary_doc_filename,
        "is_company_in_universe": 1 if cik and cik in universe_ciks else 0,
        "clean_text_chars": len(clean_text),
        "clean_text_words": count_words(clean_text),
        "section_hint_count": section_hint_count,
        "status": "ok",
        "error_message": "",
    }
    return meta, clean_text


def extract_sections_for_doc(doc_meta: dict[str, Any], clean_text: str) -> list[dict[str, Any]]:
    form = (doc_meta.get("form_type") or "").upper().strip()
    text = clean_text

    sections: list[dict[str, Any]] = []

    def slice_items(item_patterns: list[tuple[str, list[str]]], text_src: str) -> None:
        nonlocal sections
        markers = []
        for idx, (name, patterns) in enumerate(item_patterns):
            for pat in patterns:
                for m in re.finditer(pat, text_src, flags=re.IGNORECASE | re.MULTILINE):
                    markers.append((m.start(), name))
                    break
        markers = sorted(set(markers), key=lambda x: x[0])
        for i, (start, name) in enumerate(markers):
            end = markers[i + 1][0] if i + 1 < len(markers) else len(text_src)
            chunk = text_src[start:end].strip()
            if len(chunk) >= 300:
                sections.append({
                    "section_name": name,
                    "section_rank": i + 1,
                    "source_mode": "regex_item",
                    "text": chunk,
                })

    if form == "10-K":
        patterns = [
            ("business", [r"(?im)^\s*item\s+1[\.\-:\s]+business\b"]),
            ("risk_factors", [r"(?im)^\s*item\s+1A[\.\-:\s]+risk\s+factors\b"]),
            ("mda", [r"(?im)^\s*item\s+7[\.\-:\s]+management'?s?\s+discussion", r"(?im)^\s*item\s+7[\.\-:\s]+md&a"]),
        ]
        slice_items(patterns, text)

    elif form == "10-Q":
        patterns = [
            ("financial_statements", [r"(?im)^\s*item\s+1[\.\-:\s]+financial\s+statements\b"]),
            ("mda", [r"(?im)^\s*item\s+2[\.\-:\s]+management'?s?\s+discussion", r"(?im)^\s*item\s+2[\.\-:\s]+md&a"]),
            ("risk_factors", [r"(?im)^\s*item\s+1A[\.\-:\s]+risk\s+factors\b"]),
        ]
        slice_items(patterns, text)

    elif form == "8-K":
        # extract item-based segments
        markers = []
        for m in re.finditer(r"(?im)^\s*item\s+([0-9]{1,2}\.[0-9]{2})\b", text):
            item_no = m.group(1)
            markers.append((m.start(), f"item_{item_no.replace('.', '_')}"))
        markers = sorted(markers, key=lambda x: x[0])
        for i, (start, name) in enumerate(markers):
            end = markers[i + 1][0] if i + 1 < len(markers) else len(text)
            chunk = text[start:end].strip()
            if len(chunk) >= 200:
                sections.append({
                    "section_name": name,
                    "section_rank": i + 1,
                    "source_mode": "regex_item",
                    "text": chunk,
                })

    elif form == "DEF 14A":
        heading_patterns = [
            ("executive_compensation", [r"(?im)^\s*executive\s+compensation\b", r"(?im)^\s*compensation\s+discussion"]),
            ("corporate_governance", [r"(?im)^\s*corporate\s+governance\b", r"(?im)^\s*board\s+of\s+directors\b"]),
            ("security_ownership", [r"(?im)^\s*security\s+ownership\b", r"(?im)^\s*beneficial\s+ownership\b"]),
        ]
        slice_items(heading_patterns, text)

    # fallback if no sections found: split by paragraphs and keep the first large chunk
    if not sections:
        paras = [p.strip() for p in re.split(r"\n{2,}", text) if len(p.strip()) >= 250]
        if paras:
            joined = "\n\n".join(paras[:20])
            sections.append({
                "section_name": "full_document_fallback",
                "section_rank": 1,
                "source_mode": "fallback",
                "text": joined,
            })

    return sections


def merge_csv_parts(glob_pattern: str, out_path: Path, header: list[str]) -> int:
    parts = sorted(PARTS_DIR.glob(glob_pattern))
    if not parts:
        return 0
    tmp = out_path.with_suffix(out_path.suffix + ".part")
    count = 0
    with tmp.open("w", newline="", encoding="utf-8") as out_f:
        writer = csv.writer(out_f)
        writer.writerow(header)
        for i, p in enumerate(parts):
            with p.open("r", newline="", encoding="utf-8") as in_f:
                reader = csv.reader(in_f)
                # skip header
                try:
                    next(reader)
                except StopIteration:
                    continue
                for row in reader:
                    writer.writerow(row)
                    count += 1
    tmp.replace(out_path)
    return count


def build_inventory_rows(input_root: Path) -> list[list[Any]]:
    rows = []
    for p in sorted(input_root.rglob("*")):
        if not p.is_file():
            continue
        if p.suffix.lower() not in {".txt", ".html", ".htm"}:
            continue
        rel = str(p.relative_to(input_root)).replace("\\", "/")
        year, form_folder = derive_year_form(rel)
        if form_folder not in ALLOWED_FORM_FOLDERS:
            continue
        rows.append([
            rel,
            str(p),
            year,
            form_folder,
            p.stat().st_size,
            p.name,
        ])
    return rows


# ============================================================
# MODULE A — INVENTORY
# ============================================================

def run_inventory(args: argparse.Namespace) -> None:
    start = now_ts()
    rows = build_inventory_rows(Path(args.input_dir))
    write_csv(INVENTORY_CSV, INVENTORY_HEADER, rows)
    elapsed = now_ts() - start
    print(f"[inventory] rows={len(rows):,} | output={INVENTORY_CSV} | elapsed={fmt_elapsed(elapsed)}", flush=True)


# ============================================================
# MODULE B — CLEAN / PARSE / FILTER / FULL CORPUS
# ============================================================

def process_clean_shard(
    shard_id: int,
    files: list[str],
    input_root_str: str,
    universe_ciks: list[str],
    min_free_bytes: int,
) -> dict[str, Any]:
    input_root = Path(input_root_str)
    universe = set(universe_ciks)

    meta_tmp = PARTS_DIR / f"docs_meta_part_{shard_id:05d}.csv.part"
    text_tmp = PARTS_DIR / f"docs_text_part_{shard_id:05d}.jsonl.gz.part"
    err_tmp = PARTS_DIR / f"docs_errors_part_{shard_id:05d}.csv.part"

    meta_final = PARTS_DIR / f"docs_meta_part_{shard_id:05d}.csv"
    text_final = PARTS_DIR / f"docs_text_part_{shard_id:05d}.jsonl.gz"
    err_final = PARTS_DIR / f"docs_errors_part_{shard_id:05d}.csv"

    # skip if already done
    if meta_final.exists() and text_final.exists() and done_marker(meta_final).exists():
        return {
            "shard_id": shard_id,
            "input_files": 0,
            "ok_files": 0,
            "failed_files": 0,
            "text_records": 0,
            "reused": True,
        }

    ok_files = 0
    failed_files = 0
    text_records = 0

    with meta_tmp.open("w", newline="", encoding="utf-8") as meta_f, \
         gzip.open(text_tmp, "wb", compresslevel=COMPRESSLEVEL) as text_f, \
         err_tmp.open("w", newline="", encoding="utf-8") as err_f:

        meta_w = csv.writer(meta_f)
        err_w = csv.writer(err_f)
        meta_w.writerow(DOC_META_HEADER)
        err_w.writerow(ERRORS_HEADER)

        for idx, file_str in enumerate(files, start=1):
            if idx % 64 == 0:
                if free_bytes(PARTS_DIR) < min_free_bytes:
                    raise RuntimeError("LowDiskSpaceError: free disk dropped below reserve during clean stage")

            path = Path(file_str)
            rel = str(path.relative_to(input_root)).replace("\\", "/")

            try:
                raw = path.read_bytes()
                meta, clean_text = parse_raw_filing(raw, rel, universe)

                # Only keep allowed form canonical names and target universe
                canonical_form = (meta["form_type"] or "").upper().replace("_", " ")
                if canonical_form not in ALLOWED_FORMS:
                    # If SEC header says something odd but folder is target, fall back to folder form.
                    folder_form = folder_form_to_canonical(meta["folder_form"]).upper()
                    if folder_form in ALLOWED_FORMS:
                        meta["form_type"] = folder_form
                    else:
                        meta["status"] = "skipped_non_target_form"

                meta["part_id"] = f"{shard_id:05d}"

                meta_w.writerow([
                    meta["doc_id"], meta["rel_path"], meta["part_id"], meta["year"],
                    meta["form_type"], meta["folder_form"], meta["cik"], meta["entity_name"],
                    meta["filing_date"], meta["period_of_report"], meta["acceptance_datetime"],
                    meta["accession"], meta["primary_doc_type"], meta["primary_doc_filename"],
                    meta["is_company_in_universe"], meta["clean_text_chars"], meta["clean_text_words"],
                    meta["section_hint_count"], meta["status"], meta["error_message"],
                ])

                if meta["is_company_in_universe"] == 1 and meta["status"] == "ok" and clean_text:
                    rec = {
                        "doc_id": meta["doc_id"],
                        "rel_path": meta["rel_path"],
                        "part_id": meta["part_id"],
                        "year": meta["year"],
                        "form_type": meta["form_type"],
                        "cik": meta["cik"],
                        "entity_name": meta["entity_name"],
                        "filing_date": meta["filing_date"],
                        "period_of_report": meta["period_of_report"],
                        "acceptance_datetime": meta["acceptance_datetime"],
                        "accession": meta["accession"],
                        "primary_doc_type": meta["primary_doc_type"],
                        "primary_doc_filename": meta["primary_doc_filename"],
                        "clean_text": clean_text,
                    }
                    text_f.write(json_dumps(rec) + b"\n")
                    text_records += 1
                    ok_files += 1
                else:
                    ok_files += 1

            except Exception as exc:
                failed_files += 1
                doc_id = hash_id(rel)
                err_w.writerow([
                    "clean",
                    rel,
                    doc_id,
                    type(exc).__name__,
                    str(exc),
                ])
                meta_w.writerow([
                    doc_id, rel, f"{shard_id:05d}", "", "", "", "", "", "", "", "", "", "", "",
                    0, 0, 0, 0, "error", str(exc),
                ])

    meta_tmp.replace(meta_final)
    text_tmp.replace(text_final)
    err_tmp.replace(err_final)
    done_marker(meta_final).write_text("done", encoding="utf-8")

    return {
        "shard_id": shard_id,
        "input_files": len(files),
        "ok_files": ok_files,
        "failed_files": failed_files,
        "text_records": text_records,
        "reused": False,
    }


def run_clean(args: argparse.Namespace) -> None:
    start = now_ts()
    input_root = Path(args.input_dir)
    workers = choose_worker_count(args.workers)
    min_free_bytes = int(args.min_free_gb * (1024 ** 3))

    universe = load_cik_union([Path(args.cik_map), Path(args.issuer_tickers)])
    inv_rows = []
    if INVENTORY_CSV.exists():
        with INVENTORY_CSV.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            inv_rows = [row for row in reader]
    else:
        rows = build_inventory_rows(input_root)
        write_csv(INVENTORY_CSV, INVENTORY_HEADER, rows)
        inv_rows = [
            dict(zip(INVENTORY_HEADER, map(str, row)))
            for row in rows
        ]

    all_files = [Path(r["abs_path"]) for r in inv_rows]

    if args.resume:
        processed = load_processed_relpaths("docs_meta_part_*.csv", "rel_path")
        files = [p for p in all_files if str(p.relative_to(input_root)).replace("\\", "/") not in processed]
    else:
        files = all_files

    if not files:
        print("[clean] nothing to do", flush=True)
        return

    shard_count = max(1, min(len(files), workers * max(1, args.shard_multiplier)))
    shards = greedy_partition(files, shard_count)

    print(f"[clean] files={len(files):,} workers={workers} shards={len(shards)} free_disk={human_bytes(free_bytes(OUT_ROOT))}", flush=True)

    total_done = 0
    total_ok = 0
    total_fail = 0
    total_records = 0

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = []
        for i, shard_files in enumerate(shards, start=1):
            futures.append(ex.submit(
                process_clean_shard,
                i,
                [str(p) for p in shard_files],
                str(input_root),
                list(universe),
                min_free_bytes,
            ))

        completed = 0
        for fut in as_completed(futures):
            res = fut.result()
            completed += 1
            total_done += res["input_files"]
            total_ok += res["ok_files"]
            total_fail += res["failed_files"]
            total_records += res["text_records"]
            elapsed = now_ts() - start
            rate = total_done / max(elapsed, 1e-9)
            print(
                f"[clean] shards {completed}/{len(shards)} | files {total_done:,}/{len(files):,} | "
                f"ok {total_ok:,} | fail {total_fail:,} | docs {total_records:,} | rate {rate:,.1f} files/s",
                flush=True
            )

    # Merge metadata and errors; text stays partitioned
    merged_meta = merge_csv_parts("docs_meta_part_*.csv", FULL_DOCS_MANIFEST, DOC_META_HEADER)
    merged_err = merge_csv_parts("docs_errors_part_*.csv", ERRORS_CSV, ERRORS_HEADER)

    elapsed = now_ts() - start
    print(
        f"[clean] done | manifest_rows={merged_meta:,} | errors={merged_err:,} | "
        f"text_parts={len(list(PARTS_DIR.glob('docs_text_part_*.jsonl.gz'))):,} | elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# MODULE C — SECTIONS
# ============================================================

def iter_doc_text_records() -> Iterable[dict[str, Any]]:
    for p in sorted(PARTS_DIR.glob("docs_text_part_*.jsonl.gz")):
        with gzip.open(p, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                yield json.loads(line)


def process_section_part(
    shard_id: int,
    records: list[dict[str, Any]],
    min_free_bytes: int,
) -> dict[str, Any]:
    meta_tmp = PARTS_DIR / f"sections_meta_part_{shard_id:05d}.csv.part"
    text_tmp = PARTS_DIR / f"sections_text_part_{shard_id:05d}.jsonl.gz.part"
    err_tmp = PARTS_DIR / f"sections_errors_part_{shard_id:05d}.csv.part"

    meta_final = PARTS_DIR / f"sections_meta_part_{shard_id:05d}.csv"
    text_final = PARTS_DIR / f"sections_text_part_{shard_id:05d}.jsonl.gz"
    err_final = PARTS_DIR / f"sections_errors_part_{shard_id:05d}.csv"

    if meta_final.exists() and text_final.exists() and done_marker(meta_final).exists():
        return {
            "shard_id": shard_id,
            "input_docs": 0,
            "section_rows": 0,
            "failed_docs": 0,
            "reused": True,
        }

    section_rows = 0
    failed_docs = 0

    with meta_tmp.open("w", newline="", encoding="utf-8") as meta_f, \
         gzip.open(text_tmp, "wb", compresslevel=COMPRESSLEVEL) as text_f, \
         err_tmp.open("w", newline="", encoding="utf-8") as err_f:

        meta_w = csv.writer(meta_f)
        err_w = csv.writer(err_f)
        meta_w.writerow(SECTION_META_HEADER)
        err_w.writerow(ERRORS_HEADER)

        for idx, rec in enumerate(records, start=1):
            if idx % 64 == 0:
                if free_bytes(PARTS_DIR) < min_free_bytes:
                    raise RuntimeError("LowDiskSpaceError: free disk dropped below reserve during sections stage")

            try:
                doc_meta = {
                    "doc_id": rec["doc_id"],
                    "part_id": rec["part_id"],
                    "year": rec["year"],
                    "form_type": rec["form_type"],
                    "cik": rec["cik"],
                    "filing_date": rec["filing_date"],
                    "accession": rec["accession"],
                }
                sections = extract_sections_for_doc(doc_meta, rec["clean_text"])

                for s in sections:
                    sid = hash_id(rec["doc_id"], s["section_name"], str(s["section_rank"]))
                    text_chars = len(s["text"])
                    text_words = count_words(s["text"])
                    meta_w.writerow([
                        sid,
                        rec["doc_id"],
                        f"{shard_id:05d}",
                        rec["year"],
                        rec["form_type"],
                        rec["cik"],
                        rec["filing_date"],
                        rec["accession"],
                        s["section_name"],
                        s["section_rank"],
                        s["source_mode"],
                        text_chars,
                        text_words,
                    ])
                    text_f.write(json_dumps({
                        "section_id": sid,
                        "doc_id": rec["doc_id"],
                        "part_id": f"{shard_id:05d}",
                        "year": rec["year"],
                        "form_type": rec["form_type"],
                        "cik": rec["cik"],
                        "filing_date": rec["filing_date"],
                        "accession": rec["accession"],
                        "section_name": s["section_name"],
                        "section_rank": s["section_rank"],
                        "source_mode": s["source_mode"],
                        "text": s["text"],
                    }) + b"\n")
                    section_rows += 1

            except Exception as exc:
                failed_docs += 1
                err_w.writerow([
                    "sections",
                    rec.get("rel_path", ""),
                    rec.get("doc_id", ""),
                    type(exc).__name__,
                    str(exc),
                ])

    meta_tmp.replace(meta_final)
    text_tmp.replace(text_final)
    err_tmp.replace(err_final)
    done_marker(meta_final).write_text("done", encoding="utf-8")

    return {
        "shard_id": shard_id,
        "input_docs": len(records),
        "section_rows": section_rows,
        "failed_docs": failed_docs,
        "reused": False,
    }


def run_sections(args: argparse.Namespace) -> None:
    start = now_ts()
    workers = choose_worker_count(args.workers)
    min_free_bytes = int(args.min_free_gb * (1024 ** 3))

    # group records into shards without rereading raw filings
    docs = list(iter_doc_text_records())
    if args.resume:
        processed_doc_ids = set()
        for p in PARTS_DIR.glob("sections_meta_part_*.csv"):
            try:
                with p.open("r", newline="", encoding="utf-8") as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        did = row.get("doc_id", "")
                        if did:
                            processed_doc_ids.add(did)
            except Exception:
                continue
        docs = [d for d in docs if d["doc_id"] not in processed_doc_ids]

    if not docs:
        print("[sections] nothing to do", flush=True)
        return

    shard_count = max(1, min(len(docs), workers * max(1, args.shard_multiplier)))
    # simple contiguous chunking (docs already loaded; minimal extra I/O)
    chunk_size = math.ceil(len(docs) / shard_count)
    shards = [docs[i:i + chunk_size] for i in range(0, len(docs), chunk_size)]

    total_docs = 0
    total_sections = 0
    total_fail = 0

    print(f"[sections] docs={len(docs):,} workers={workers} shards={len(shards)}", flush=True)

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = []
        for i, shard in enumerate(shards, start=1):
            futures.append(ex.submit(process_section_part, i, shard, min_free_bytes))

        completed = 0
        for fut in as_completed(futures):
            res = fut.result()
            completed += 1
            total_docs += res["input_docs"]
            total_sections += res["section_rows"]
            total_fail += res["failed_docs"]
            elapsed = now_ts() - start
            rate = total_docs / max(elapsed, 1e-9)
            print(
                f"[sections] shards {completed}/{len(shards)} | docs {total_docs:,}/{len(docs):,} | "
                f"sections {total_sections:,} | fail_docs {total_fail:,} | rate {rate:,.1f} docs/s",
                flush=True
            )

    merged_meta = merge_csv_parts("sections_meta_part_*.csv", FULL_SECTIONS_MANIFEST, SECTION_META_HEADER)
    merge_csv_parts("sections_errors_part_*.csv", ERRORS_CSV, ERRORS_HEADER)

    elapsed = now_ts() - start
    print(
        f"[sections] done | section_rows={merged_meta:,} | text_parts={len(list(PARTS_DIR.glob('sections_text_part_*.jsonl.gz'))):,} "
        f"| elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# MODULE D — QUALITY / COVERAGE REPORTS
# ============================================================

def read_csv_dicts(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def run_quality(args: argparse.Namespace) -> None:
    start = now_ts()
    docs = read_csv_dicts(FULL_DOCS_MANIFEST)
    sections = read_csv_dicts(FULL_SECTIONS_MANIFEST)

    docs_kept = [d for d in docs if d.get("status") == "ok" and d.get("is_company_in_universe") == "1"]
    if not docs_kept:
        raise RuntimeError("No full docs manifest rows found. Run clean first.")

    by_year = Counter()
    by_year_form = Counter()
    by_year_company = Counter()
    by_section = Counter()
    by_doc_section_presence = Counter()

    duplicate_accessions = Counter()
    length_stats = defaultdict(list)

    for d in docs_kept:
        year = d["year"]
        form = d["form_type"]
        cik = d["cik"]
        by_year[year] += 1
        by_year_form[(year, form)] += 1
        by_year_company[(year, cik)] += 1
        if d.get("accession"):
            duplicate_accessions[(d["cik"], d["accession"])] += 1
        try:
            length_stats[form].append(int(d["clean_text_words"] or 0))
        except Exception:
            pass

    sections_by_doc = defaultdict(set)
    for s in sections:
        did = s["doc_id"]
        sec_name = s["section_name"]
        sections_by_doc[did].add(sec_name)
        by_section[(s["year"], s["form_type"], sec_name)] += 1

    for did, secs in sections_by_doc.items():
        by_doc_section_presence[len(secs)] += 1

    write_csv(
        YEAR_COVERAGE_CSV,
        ["year", "doc_count"],
        [[y, c] for y, c in sorted(by_year.items())]
    )

    write_csv(
        YEAR_FORM_COVERAGE_CSV,
        ["year", "form_type", "doc_count"],
        [[y, f, c] for (y, f), c in sorted(by_year_form.items())]
    )

    write_csv(
        YEAR_COMPANY_COVERAGE_CSV,
        ["year", "cik", "doc_count"],
        [[y, cik, c] for (y, cik), c in sorted(by_year_company.items())]
    )

    write_csv(
        SECTION_COVERAGE_CSV,
        ["year", "form_type", "section_name", "section_count"],
        [[y, f, s, c] for (y, f, s), c in sorted(by_section.items())]
    )

    summary = {
        "docs_total": len(docs),
        "docs_kept_full_filtered": len(docs_kept),
        "sections_total": len(sections),
        "years_covered": sorted(by_year.keys()),
        "duplicate_accession_pairs_gt1": sum(1 for _, v in duplicate_accessions.items() if v > 1),
        "forms": {
            form: {
                "docs": sum(1 for d in docs_kept if d["form_type"] == form),
                "avg_words": (sum(length_stats[form]) / len(length_stats[form])) if length_stats[form] else 0.0,
                "median_words_approx": sorted(length_stats[form])[len(length_stats[form]) // 2] if length_stats[form] else 0,
            }
            for form in FORM_ORDER
        },
        "year_counts": dict(sorted(by_year.items())),
        "section_presence_by_num_sections_per_doc": dict(sorted(by_doc_section_presence.items())),
    }

    with QUALITY_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    elapsed = now_ts() - start
    print(f"[quality] done | docs={len(docs_kept):,} | sections={len(sections):,} | elapsed={fmt_elapsed(elapsed)}", flush=True)


# ============================================================
# MODULE E — FULL + BALANCED DATASETS
# ============================================================

def weighted_form_allocation(target_n: int, available_by_form: dict[str, int], weights: dict[str, float]) -> dict[str, int]:
    quotas = {f: 0 for f in FORM_ORDER}
    remaining = target_n

    # initial weighted allocation
    for f in FORM_ORDER:
        want = int(round(target_n * weights.get(f, 0.0)))
        take = min(want, available_by_form.get(f, 0))
        quotas[f] = take
        remaining -= take

    if remaining <= 0:
        return quotas

    # fill remainder from forms with spare capacity, prioritized by weight then availability
    forms = sorted(FORM_ORDER, key=lambda x: (weights.get(x, 0.0), available_by_form.get(x, 0)), reverse=True)
    while remaining > 0:
        progressed = False
        for f in forms:
            if quotas[f] < available_by_form.get(f, 0):
                quotas[f] += 1
                remaining -= 1
                progressed = True
                if remaining <= 0:
                    break
        if not progressed:
            break
    return quotas


def run_datasets(args: argparse.Namespace) -> None:
    start = now_ts()
    rng = random.Random(args.seed)

    docs = read_csv_dicts(FULL_DOCS_MANIFEST)
    sections = read_csv_dicts(FULL_SECTIONS_MANIFEST)

    docs_kept = [d for d in docs if d.get("status") == "ok" and d.get("is_company_in_universe") == "1"]
    if not docs_kept:
        raise RuntimeError("No full docs manifest rows found. Run clean first.")

    # full dataset manifests (already filtered by universe)
    write_csv(FULL_DOCS_MANIFEST, DOC_META_HEADER, [
        [d.get(col, "") for col in DOC_META_HEADER] for d in docs_kept
    ])

    sections_kept = [s for s in sections if s.get("doc_id") in {d["doc_id"] for d in docs_kept}]
    write_csv(FULL_SECTIONS_MANIFEST, SECTION_META_HEADER, [
        [s.get(col, "") for col in SECTION_META_HEADER] for s in sections_kept
    ])

    by_year = Counter(d["year"] for d in docs_kept if d.get("year"))
    usable_years = [y for y, c in sorted(by_year.items()) if c >= args.min_docs_per_year]
    if not usable_years:
        raise RuntimeError("No usable years satisfy min_docs_per_year; lower the threshold or inspect coverage.")

    weakest_usable_year = min(usable_years, key=lambda y: by_year[y])
    target_per_year = by_year[weakest_usable_year]

    # optional override
    if args.target_per_year > 0:
        target_per_year = min(target_per_year, args.target_per_year)

    weights = DEFAULT_FORM_WEIGHTS.copy()
    if args.form_weights:
        for kv in args.form_weights.split(","):
            if "=" not in kv:
                continue
            k, v = kv.split("=", 1)
            k = k.strip().upper().replace("_", " ")
            if k in weights:
                weights[k] = float(v)

    # normalize weights
    total_w = sum(weights.values()) or 1.0
    weights = {k: v / total_w for k, v in weights.items()}

    docs_by_year_form: dict[str, dict[str, list[dict[str, str]]]] = defaultdict(lambda: defaultdict(list))
    for d in docs_kept:
        if d["year"] in usable_years:
            docs_by_year_form[d["year"]][d["form_type"]].append(d)

    sampled_doc_ids: set[str] = set()
    balanced_rows: list[dict[str, str]] = []
    sampling_audit: list[dict[str, Any]] = []

    for year in sorted(usable_years):
        available_by_form = {f: len(docs_by_year_form[year].get(f, [])) for f in FORM_ORDER}
        quotas = weighted_form_allocation(target_per_year, available_by_form, weights)

        for form in FORM_ORDER:
            candidates = docs_by_year_form[year].get(form, [])
            # optional company cap
            if args.max_docs_per_cik_per_year_form > 0:
                by_cik = defaultdict(list)
                for d in candidates:
                    by_cik[d["cik"]].append(d)
                capped = []
                for cik, rows in by_cik.items():
                    rows = rows[:]
                    rng.shuffle(rows)
                    capped.extend(rows[:args.max_docs_per_cik_per_year_form])
                candidates = capped

            rng.shuffle(candidates)
            chosen = candidates[:quotas.get(form, 0)]
            for d in chosen:
                sampled_doc_ids.add(d["doc_id"])
                balanced_rows.append(d)

        sampling_audit.append({
            "year": year,
            "available_total": by_year[year],
            "target_total": target_per_year,
            "available_by_form": available_by_form,
            "quotas": quotas,
        })

    balanced_sections = [s for s in sections_kept if s["doc_id"] in sampled_doc_ids]

    write_csv(BALANCED_DOCS_MANIFEST, DOC_META_HEADER, [
        [d.get(col, "") for col in DOC_META_HEADER] for d in balanced_rows
    ])
    write_csv(BALANCED_SECTIONS_MANIFEST, SECTION_META_HEADER, [
        [s.get(col, "") for col in SECTION_META_HEADER] for s in balanced_sections
    ])

    with BALANCE_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump({
            "seed": args.seed,
            "min_docs_per_year": args.min_docs_per_year,
            "usable_years": usable_years,
            "weakest_usable_year": weakest_usable_year,
            "target_per_year": target_per_year,
            "form_weights": weights,
            "balanced_doc_count": len(balanced_rows),
            "balanced_section_count": len(balanced_sections),
            "sampling_audit": sampling_audit,
        }, f, indent=2)

    elapsed = now_ts() - start
    print(
        f"[datasets] done | usable_years={len(usable_years):,} | target_per_year={target_per_year:,} | "
        f"balanced_docs={len(balanced_rows):,} | balanced_sections={len(balanced_sections):,} | elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# MODULE F — FINBERT-READY CHUNKS
# ============================================================

SECTION_PRIORITY = {
    "risk_factors": 1,
    "mda": 2,
    "business": 3,
    "financial_statements": 4,
    "executive_compensation": 5,
    "corporate_governance": 6,
    "security_ownership": 7,
    "full_document_fallback": 99,
}


def iter_section_text_records() -> Iterable[dict[str, Any]]:
    for p in sorted(PARTS_DIR.glob("sections_text_part_*.jsonl.gz")):
        with gzip.open(p, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                yield json.loads(line)


def load_doc_text_map() -> dict[str, dict[str, Any]]:
    m = {}
    for p in sorted(PARTS_DIR.glob("docs_text_part_*.jsonl.gz")):
        with gzip.open(p, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                m[rec["doc_id"]] = rec
    return m


def chunk_words(text: str, chunk_size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    i = 0
    step = max(1, chunk_size - overlap)
    while i < len(words):
        chunk = words[i:i + chunk_size]
        if len(chunk) >= max(50, chunk_size // 3):
            chunks.append(" ".join(chunk))
        i += step
    return chunks


def run_finbert(args: argparse.Namespace) -> None:
    start = now_ts()
    full_docs = read_csv_dicts(FULL_DOCS_MANIFEST)
    balanced_docs = read_csv_dicts(BALANCED_DOCS_MANIFEST)

    full_doc_ids = {d["doc_id"] for d in full_docs}
    balanced_doc_ids = {d["doc_id"] for d in balanced_docs}

    doc_text_map = load_doc_text_map()
    sections = list(iter_section_text_records())

    # group sections by doc with priority
    sec_by_doc = defaultdict(list)
    for s in sections:
        if s["doc_id"] in full_doc_ids:
            sec_by_doc[s["doc_id"]].append(s)

    def build_rows(target_doc_ids: set[str]) -> list[list[Any]]:
        rows = []
        for doc_id in sorted(target_doc_ids):
            doc = doc_text_map.get(doc_id)
            if not doc:
                continue

            # choose prioritized section texts if available, else fallback to full doc
            sources = sorted(
                sec_by_doc.get(doc_id, []),
                key=lambda x: (SECTION_PRIORITY.get(x["section_name"], 50), int(x.get("section_rank", 999)))
            )

            text_units = []
            if sources:
                for s in sources[:args.max_sections_per_doc]:
                    if len(s["text"]) >= args.min_section_chars:
                        text_units.append((s["section_name"], s["text"]))
            if not text_units:
                text_units.append(("full_document", doc["clean_text"]))

            chunk_idx = 0
            for source_name, source_text in text_units:
                for ch in chunk_words(source_text, args.chunk_words, args.overlap_words):
                    chunk_idx += 1
                    chunk_id = hash_id(doc_id, source_name, str(chunk_idx))
                    rows.append([
                        chunk_id,
                        doc_id,
                        doc["year"],
                        doc["form_type"],
                        doc["cik"],
                        doc["filing_date"],
                        doc["accession"],
                        source_name,
                        chunk_idx,
                        count_words(ch),
                        ch,
                    ])
        return rows

    header = [
        "chunk_id",
        "doc_id",
        "year",
        "form_type",
        "cik",
        "filing_date",
        "accession",
        "source_name",
        "chunk_index",
        "word_count",
        "text",
    ]

    all_rows = build_rows(full_doc_ids)
    bal_rows = [r for r in all_rows if r[1] in balanced_doc_ids]

    write_csv(FINBERT_CHUNKS_ALL_CSV, header, all_rows)
    write_csv(FINBERT_CHUNKS_BALANCED_CSV, header, bal_rows)

    with FINBERT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump({
            "chunk_words": args.chunk_words,
            "overlap_words": args.overlap_words,
            "max_sections_per_doc": args.max_sections_per_doc,
            "min_section_chars": args.min_section_chars,
            "all_chunk_count": len(all_rows),
            "balanced_chunk_count": len(bal_rows),
        }, f, indent=2)

    elapsed = now_ts() - start
    print(
        f"[finbert] done | all_chunks={len(all_rows):,} | balanced_chunks={len(bal_rows):,} | elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# ALL
# ============================================================

def run_all(args: argparse.Namespace) -> None:
    if not INVENTORY_CSV.exists() or args.force_inventory:
        run_inventory(args)
    run_clean(args)
    run_sections(args)
    run_quality(args)
    run_datasets(args)
    run_finbert(args)


# ============================================================
# CLI
# ============================================================

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Comprehensive HDD-aware SEC filings text cleaning + engineering pipeline."
    )
    sub = p.add_subparsers(dest="command", required=True)

    def common(sp):
        sp.add_argument("--input-dir", type=str, default=str(RAW_FILINGS_DIR))
        sp.add_argument("--cik-map", type=str, default=str(DEFAULT_CIK_MAP))
        sp.add_argument("--issuer-tickers", type=str, default=str(DEFAULT_ISSUER_TICKERS))
        sp.add_argument("--workers", type=int, default=2)
        sp.add_argument("--shard-multiplier", type=int, default=0)
        sp.add_argument("--resume", action="store_true")
        sp.add_argument("--overwrite", action="store_true")
        sp.add_argument("--min-free-gb", type=float, default=8.0)

    s = sub.add_parser("inventory")
    common(s)

    s = sub.add_parser("clean")
    common(s)

    s = sub.add_parser("sections")
    common(s)

    s = sub.add_parser("quality")
    common(s)

    s = sub.add_parser("datasets")
    common(s)
    s.add_argument("--seed", type=int, default=42)
    s.add_argument("--min-docs-per-year", type=int, default=1000)
    s.add_argument("--target-per-year", type=int, default=0)
    s.add_argument("--max-docs-per-cik-per-year-form", type=int, default=5)
    s.add_argument("--form-weights", type=str, default="")

    s = sub.add_parser("finbert")
    common(s)
    s.add_argument("--chunk-words", type=int, default=350)
    s.add_argument("--overlap-words", type=int, default=50)
    s.add_argument("--max-sections-per-doc", type=int, default=3)
    s.add_argument("--min-section-chars", type=int, default=500)

    s = sub.add_parser("all")
    common(s)
    s.add_argument("--force-inventory", action="store_true")
    s.add_argument("--seed", type=int, default=42)
    s.add_argument("--min-docs-per-year", type=int, default=1000)
    s.add_argument("--target-per-year", type=int, default=0)
    s.add_argument("--max-docs-per-cik-per-year-form", type=int, default=5)
    s.add_argument("--form-weights", type=str, default="")
    s.add_argument("--chunk-words", type=int, default=350)
    s.add_argument("--overlap-words", type=int, default=50)
    s.add_argument("--max-sections-per-doc", type=int, default=3)
    s.add_argument("--min-section-chars", type=int, default=500)

    return p.parse_args()


def main() -> None:
    args = parse_args()
    ensure_dirs(args.overwrite)

    print(f"Input directory : {args.input_dir}", flush=True)
    print(f"Output root     : {OUT_ROOT}", flush=True)
    print(f"Workers         : {choose_worker_count(args.workers)}", flush=True)
    print(f"Resume mode     : {args.resume}", flush=True)
    print(f"Min free space  : {args.min_free_gb:.2f} GB", flush=True)
    print(f"JSON parser     : {'orjson' if HAS_ORJSON else 'stdlib json'}", flush=True)

    cmd = args.command
    if cmd == "inventory":
        run_inventory(args)
    elif cmd == "clean":
        run_clean(args)
    elif cmd == "sections":
        run_sections(args)
    elif cmd == "quality":
        run_quality(args)
    elif cmd == "datasets":
        run_datasets(args)
    elif cmd == "finbert":
        run_finbert(args)
    elif cmd == "all":
        run_all(args)
    else:
        raise SystemExit(f"Unknown command: {cmd}")


if __name__ == "__main__":
    main()

'''
It is built around your current reality: a **partial, 
storage-constrained downloaded filings corpus** rather than full SEC coverage, 
and it prepares the text stream for the FinBERT setup.   

What's inside:

* **A. inventory**
* **B. clean / parse / filter / full corpus**
* **C. section extraction**
* **D. quality + coverage reports**
* **E. full + balanced dataset manifests**
* **F. FinBERT-ready chunk generation**

It is designed to:

* run modules **selectively**
* support **resume**
* use **partitioned outputs**
* keep I/O lower for **HDD-backed storage**
* avoid rereading raw filings more than necessary
* build both:

  * a **full cleaned filtered corpus**
  * a **balanced modeling corpus**

Typical usage:
python data/sec_filings_text_pipeline.py inventory
python data/sec_filings_text_pipeline.py clean --resume
python data/sec_filings_text_pipeline.py sections --resume
python data/sec_filings_text_pipeline.py quality
python data/sec_filings_text_pipeline.py datasets
python data/sec_filings_text_pipeline.py finbert
python data/sec_filings_text_pipeline.py all --resume
'''


### data\sec_filings_text_pipeline_v2.py

In [ ]:
#!/usr/bin/env python3
"""
sec_filings_text_pipeline_v2.py

Linux-targeted end-to-end text engineering pipeline for a downloaded SEC filings corpus.
Designed for storage-constrained, partially downloaded corpora and HDD-backed storage.

MODULES
A. inventory   -> Scan downloaded filings on disk and build raw inventory
B. clean       -> Parse metadata + extract/clean primary text + build full filtered corpus
C. sections    -> Extract section-level text
D. quality     -> Coverage + quality reports
E. datasets    -> Build full/balanced modeling manifests
F. finbert     -> Build FinBERT-ready chunk datasets

Run modules selectively:
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py inventory
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py clean --resume
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py sections --resume
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py quality
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py datasets
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py finbert
    nice -n -20 python -u data/sec_filings_text_pipeline_v2.py all --resume

Key design decisions:
- Minimal repeated I/O on HDD: parse each raw filing once in module B.
- Large text outputs are partitioned and gzip-compressed JSONL.
- Resume-friendly: completed parts are marked and never reprocessed.
- Company-universe filtering uses union of cleaned SEC universe CSVs.
- Balanced dataset is manifest-based (no text duplication).
"""

from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import heapq
import html
import io
import json
import math
import os
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Iterable

try:
    import orjson  # type: ignore
    HAS_ORJSON = True
except Exception:
    orjson = None
    HAS_ORJSON = False


# ============================================================
# GLOBAL PATHS / CONFIG
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

RAW_FILINGS_DIR = dataPath / "sec_edgar" / "raw" / "filings_txt"
CLEANED_DIR = dataPath / "sec_edgar" / "processed" / "cleaned"

DEFAULT_CIK_MAP = CLEANED_DIR / "cik_ticker_map_cleaned.csv"
DEFAULT_ISSUER_TICKERS = CLEANED_DIR / "issuer_master_onlyTickers.csv"

OUT_ROOT = dataPath / "sec_edgar" / "processed" / "filings_text"
PARTS_DIR = OUT_ROOT / "parts"
TMP_DIR = OUT_ROOT / "_tmp"

INVENTORY_CSV = OUT_ROOT / "filings_downloaded_inventory.csv"
FULL_DOCS_MANIFEST = OUT_ROOT / "filings_filtered_full.csv"
FULL_SECTIONS_MANIFEST = OUT_ROOT / "filings_sections_full.csv"

YEAR_COVERAGE_CSV = OUT_ROOT / "filings_year_coverage.csv"
YEAR_FORM_COVERAGE_CSV = OUT_ROOT / "filings_year_form_coverage.csv"
YEAR_COMPANY_COVERAGE_CSV = OUT_ROOT / "filings_year_company_coverage.csv"
SECTION_COVERAGE_CSV = OUT_ROOT / "filings_section_coverage.csv"
QUALITY_SUMMARY_JSON = OUT_ROOT / "filings_quality_summary.json"

BALANCED_DOCS_MANIFEST = OUT_ROOT / "filings_filtered_balanced.csv"
BALANCED_SECTIONS_MANIFEST = OUT_ROOT / "filings_sections_balanced.csv"
BALANCE_SUMMARY_JSON = OUT_ROOT / "filings_balance_summary.json"

FINBERT_CHUNKS_ALL_CSV = OUT_ROOT / "filings_finbert_chunks_all.csv"
FINBERT_CHUNKS_BALANCED_CSV = OUT_ROOT / "filings_finbert_chunks_balanced.csv"
FINBERT_SUMMARY_JSON = OUT_ROOT / "filings_finbert_summary.json"

LOG_DIR = dataPath / "sec_edgar" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
ERRORS_CSV = OUT_ROOT / "filings_text_errors.csv"

ALLOWED_FORMS = {"10-K", "10-Q", "8-K", "DEF 14A"}
ALLOWED_FORM_FOLDERS = {"10-K", "10-Q", "8-K", "DEF_14A"}

COMPRESSLEVEL = 1

INVENTORY_HEADER = [
    "rel_path",
    "abs_path",
    "year",
    "form_folder",
    "file_size_bytes",
    "file_name",
]

DOC_META_HEADER = [
    "doc_id",
    "rel_path",
    "part_id",
    "year",
    "form_type",
    "folder_form",
    "cik",
    "entity_name",
    "filing_date",
    "period_of_report",
    "acceptance_datetime",
    "accession",
    "primary_doc_type",
    "primary_doc_filename",
    "is_company_in_universe",
    "clean_text_chars",
    "clean_text_words",
    "section_hint_count",
    "status",
    "error_message",
]

SECTION_META_HEADER = [
    "section_id",
    "doc_id",
    "part_id",
    "year",
    "form_type",
    "cik",
    "filing_date",
    "accession",
    "section_name",
    "section_rank",
    "source_mode",
    "text_chars",
    "text_words",
]

ERRORS_HEADER = [
    "stage",
    "rel_path",
    "doc_id",
    "error_type",
    "error_message",
]

FORM_ORDER = ["10-K", "10-Q", "8-K", "DEF 14A"]

DEFAULT_FORM_WEIGHTS = {
    "10-K": 0.30,
    "10-Q": 0.30,
    "8-K": 0.30,
    "DEF 14A": 0.10,
}


# ============================================================
# HELPERS
# ============================================================

def json_dumps(obj: Any) -> bytes:
    if HAS_ORJSON:
        return orjson.dumps(obj)
    return json.dumps(obj, ensure_ascii=False).encode("utf-8")


def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for u in units:
        if n < 1024 or u == units[-1]:
            return f"{n:.2f} {u}"
        n /= 1024
    return f"{n:.2f} B"


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60
    return f"{hours:.2f}h"


def now_ts() -> float:
    return time.time()


def free_bytes(path: Path) -> int:
    return shutil.disk_usage(path).free


def choose_worker_count(requested: int) -> int:
    if requested > 0:
        return requested
    cpu = os.cpu_count() or 4
    return max(1, min(8, cpu // 2 if cpu >= 4 else 1))


def normalize_cik(value: Any) -> str:
    s = str(value).strip()
    if not s:
        return ""
    s = s.lstrip("0")
    return s if s else "0"


def detect_cik_column(fieldnames: list[str]) -> str:
    lowered = {x.lower(): x for x in fieldnames}
    for name in ("cik", "padded_cik", "sec_cik", "issuer_cik"):
        if name in lowered:
            return lowered[name]
    raise RuntimeError(f"Could not find CIK column in {fieldnames}")


def load_cik_union(csv_paths: list[Path]) -> set[str]:
    cik_set: set[str] = set()
    for path in csv_paths:
        if not path.exists():
            raise FileNotFoundError(path)
        with path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames or []
            cik_col = detect_cik_column(fieldnames)
            n = 0
            for row in reader:
                cik = normalize_cik(row.get(cik_col, ""))
                if cik:
                    cik_set.add(cik)
                    n += 1
            print(f"[cik-union] loaded {n:,} rows from {path} | cumulative unique CIKs={len(cik_set):,}", flush=True)
    return cik_set


def ensure_dirs(overwrite: bool) -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    PARTS_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    if overwrite:
        # only remove stage outputs, not raw data
        targets = [
            INVENTORY_CSV, FULL_DOCS_MANIFEST, FULL_SECTIONS_MANIFEST,
            YEAR_COVERAGE_CSV, YEAR_FORM_COVERAGE_CSV, YEAR_COMPANY_COVERAGE_CSV,
            SECTION_COVERAGE_CSV, QUALITY_SUMMARY_JSON,
            BALANCED_DOCS_MANIFEST, BALANCED_SECTIONS_MANIFEST, BALANCE_SUMMARY_JSON,
            FINBERT_CHUNKS_ALL_CSV, FINBERT_CHUNKS_BALANCED_CSV, FINBERT_SUMMARY_JSON,
            ERRORS_CSV,
        ]
        for t in targets:
            if t.exists():
                t.unlink()
        if PARTS_DIR.exists():
            for p in PARTS_DIR.glob("*"):
                if p.is_file():
                    p.unlink()


def write_csv(path: Path, header: list[str], rows: Iterable[list[Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".part")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for row in rows:
            w.writerow(row)
    tmp.replace(path)


def append_csv_row(path: Path, header: list[str], row: list[Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(header)
        w.writerow(row)


def greedy_partition(files: list[Path], shard_count: int) -> list[list[Path]]:
    shard_count = max(1, min(shard_count, len(files)))
    shards = [[] for _ in range(shard_count)]
    heap = [(0, i) for i in range(shard_count)]
    heapq.heapify(heap)
    sized = []
    for p in files:
        try:
            sz = p.stat().st_size
        except Exception:
            sz = 0
        sized.append((sz, p))
    sized.sort(key=lambda x: x[0], reverse=True)
    for sz, p in sized:
        total, idx = heapq.heappop(heap)
        shards[idx].append(p)
        heapq.heappush(heap, (total + sz, idx))
    return [s for s in shards if s]


def load_processed_relpaths(pattern: str, rel_field: str) -> set[str]:
    done: set[str] = set()
    for p in PARTS_DIR.glob(pattern):
        try:
            with p.open("r", newline="", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                if rel_field not in (reader.fieldnames or []):
                    continue
                for row in reader:
                    relp = (row.get(rel_field) or "").replace("\\", "/")
                    if relp:
                        done.add(relp)
        except Exception:
            continue
    return done


def done_marker(base: Path) -> Path:
    return base.with_suffix(base.suffix + ".done")


def normalize_space(text: str) -> str:
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_tags(text: str) -> str:
    # remove scripts/styles first
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", text)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?is)<!--.*?-->", " ", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    return text


def decode_bytes(raw: bytes) -> str:
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return raw.decode(enc)
        except Exception:
            pass
    return raw.decode("utf-8", errors="replace")


def hash_id(*parts: str, n: int = 16) -> str:
    h = hashlib.blake2b(digest_size=16)
    for p in parts:
        h.update(p.encode("utf-8", errors="ignore"))
        h.update(b"\x1f")
    return h.hexdigest()[:n]


SEC_HEADER_PATTERNS = {
    "cik": [
        r"CENTRAL INDEX KEY:\s*([0-9]+)",
        r"CIK:\s*([0-9]+)",
    ],
    "entity_name": [
        r"COMPANY CONFORMED NAME:\s*(.+)",
        r"COMPANY:\s*(.+)",
        r"CONFORMED NAME:\s*(.+)",
    ],
    "form_type": [
        r"CONFORMED SUBMISSION TYPE:\s*([^\n<]+)",
        r"FORM TYPE:\s*([^\n<]+)",
    ],
    "filing_date": [
        r"FILED AS OF DATE:\s*([0-9]{8})",
        r"FILING DATE:\s*([0-9]{8})",
    ],
    "period_of_report": [
        r"CONFORMED PERIOD OF REPORT:\s*([0-9]{8})",
        r"PERIOD OF REPORT:\s*([0-9]{8})",
    ],
    "acceptance_datetime": [
        r"ACCEPTANCE-DATETIME:\s*([0-9]{14})",
    ],
    "accession": [
        r"ACCESSION NUMBER:\s*([0-9\-]+)",
    ],
}


def sec_header_search(text: str, key: str) -> str:
    for pat in SEC_HEADER_PATTERNS[key]:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return ""


def yyyymmdd_to_iso(s: str) -> str:
    if len(s) == 8 and s.isdigit():
        return f"{s[:4]}-{s[4:6]}-{s[6:8]}"
    return s


def folder_form_to_canonical(folder_name: str) -> str:
    return folder_name.replace("_", " ")


def derive_year_form(rel_path: str) -> tuple[str, str]:
    parts = rel_path.replace("\\", "/").split("/")
    # expected: YEAR/FORM/file.txt OR maybe root/YEAR/FORM/file.txt already stripped in rel path
    if len(parts) >= 3:
        year = parts[0]
        folder_form = parts[1]
        return year, folder_form
    return "", ""


def find_document_blocks(text: str) -> list[dict[str, str]]:
    """
    Parse SEC SGML-style <DOCUMENT> blocks if present.
    """
    blocks: list[dict[str, str]] = []
    pattern = re.compile(r"(?is)<DOCUMENT>(.*?)</DOCUMENT>")
    for block in pattern.findall(text):
        doc_type = ""
        filename = ""
        description = ""
        m = re.search(r"(?im)^\s*<TYPE>\s*([^\n\r]+)", block)
        if m:
            doc_type = m.group(1).strip()
        m = re.search(r"(?im)^\s*<FILENAME>\s*([^\n\r]+)", block)
        if m:
            filename = m.group(1).strip()
        m = re.search(r"(?im)^\s*<DESCRIPTION>\s*([^\n\r]+)", block)
        if m:
            description = m.group(1).strip()
        # Body starts after SGML headers; if not clear, keep full block
        body = re.sub(r"(?im)^\s*<(TYPE|SEQUENCE|FILENAME|DESCRIPTION|TEXT)>\s*[^\n\r]*", "", block)
        blocks.append({
            "type": doc_type,
            "filename": filename,
            "description": description,
            "body": body,
        })
    return blocks


def choose_primary_document(full_text: str, declared_form: str) -> tuple[str, str, str]:
    """
    Returns: body, doc_type, doc_filename
    """
    blocks = find_document_blocks(full_text)
    if not blocks:
        return full_text, "", ""

    declared = declared_form.upper().strip()
    normalized_candidates = {
        declared,
        declared.replace("/A", ""),
        declared.replace(" ", ""),
        declared.replace(" ", "_"),
    }

    # best: exact or near-exact type match to declared form
    for b in blocks:
        t = b["type"].upper().strip()
        if t in normalized_candidates or t.replace(" ", "") in normalized_candidates:
            return b["body"], b["type"], b["filename"]

    # next: choose longest non-exhibit-like document
    good = []
    for b in blocks:
        t = b["type"].upper().strip()
        if t.startswith("EX-"):
            continue
        if t in {"GRAPHIC", "XML", "ZIP", "PDF"}:
            continue
        good.append(b)

    if good:
        best = max(good, key=lambda x: len(x["body"]))
        return best["body"], best["type"], best["filename"]

    best = max(blocks, key=lambda x: len(x["body"]))
    return best["body"], best["type"], best["filename"]


def clean_primary_text(body: str) -> str:
    text = html.unescape(body)
    text = strip_tags(text)
    # normalize SGML remnants
    text = re.sub(r"(?im)^\s*<[^>\n]+>\s*$", " ", text)
    text = re.sub(r"[\u00a0\u200b]+", " ", text)
    text = normalize_space(text)
    return text


def count_words(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\b\w+\b", text))


def parse_raw_filing(raw: bytes, rel_path: str, universe_ciks: set[str]) -> tuple[dict[str, Any], str]:
    full_text = decode_bytes(raw)
    header_window = full_text[:250_000]

    year, folder_form = derive_year_form(rel_path)

    cik = normalize_cik(sec_header_search(header_window, "cik"))
    entity_name = sec_header_search(header_window, "entity_name")
    form_type = sec_header_search(header_window, "form_type") or folder_form_to_canonical(folder_form)
    filing_date = yyyymmdd_to_iso(sec_header_search(header_window, "filing_date"))
    period_of_report = yyyymmdd_to_iso(sec_header_search(header_window, "period_of_report"))
    acceptance_datetime = sec_header_search(header_window, "acceptance_datetime")
    accession = sec_header_search(header_window, "accession")

    primary_body, primary_doc_type, primary_doc_filename = choose_primary_document(full_text, form_type)
    clean_text = clean_primary_text(primary_body)

    doc_id = hash_id(rel_path, accession or "", cik or "", form_type or "")
    section_hint_count = len(re.findall(r"(?im)\bitem\s+[0-9]{1,2}[a-z]?(?:\.[0-9]{2})?\b", clean_text))

    meta = {
        "doc_id": doc_id,
        "rel_path": rel_path,
        "year": year,
        "form_type": form_type,
        "folder_form": folder_form,
        "cik": cik,
        "entity_name": entity_name,
        "filing_date": filing_date,
        "period_of_report": period_of_report,
        "acceptance_datetime": acceptance_datetime,
        "accession": accession,
        "primary_doc_type": primary_doc_type,
        "primary_doc_filename": primary_doc_filename,
        "is_company_in_universe": 1 if cik and cik in universe_ciks else 0,
        "clean_text_chars": len(clean_text),
        "clean_text_words": count_words(clean_text),
        "section_hint_count": section_hint_count,
        "status": "ok",
        "error_message": "",
    }
    return meta, clean_text


def extract_sections_for_doc(doc_meta: dict[str, Any], clean_text: str) -> list[dict[str, Any]]:
    form = (doc_meta.get("form_type") or "").upper().strip()
    text = clean_text

    sections: list[dict[str, Any]] = []

    def slice_items(item_patterns: list[tuple[str, list[str]]], text_src: str) -> None:
        nonlocal sections
        markers = []
        for idx, (name, patterns) in enumerate(item_patterns):
            for pat in patterns:
                for m in re.finditer(pat, text_src, flags=re.IGNORECASE | re.MULTILINE):
                    markers.append((m.start(), name))
                    break
        markers = sorted(set(markers), key=lambda x: x[0])
        for i, (start, name) in enumerate(markers):
            end = markers[i + 1][0] if i + 1 < len(markers) else len(text_src)
            chunk = text_src[start:end].strip()
            if len(chunk) >= 300:
                sections.append({
                    "section_name": name,
                    "section_rank": i + 1,
                    "source_mode": "regex_item",
                    "text": chunk,
                })

    if form == "10-K":
        patterns = [
            ("business", [r"(?im)^\s*item\s+1[\.\-:\s]+business\b"]),
            ("risk_factors", [r"(?im)^\s*item\s+1A[\.\-:\s]+risk\s+factors\b"]),
            ("mda", [r"(?im)^\s*item\s+7[\.\-:\s]+management'?s?\s+discussion", r"(?im)^\s*item\s+7[\.\-:\s]+md&a"]),
        ]
        slice_items(patterns, text)

    elif form == "10-Q":
        patterns = [
            ("financial_statements", [r"(?im)^\s*item\s+1[\.\-:\s]+financial\s+statements\b"]),
            ("mda", [r"(?im)^\s*item\s+2[\.\-:\s]+management'?s?\s+discussion", r"(?im)^\s*item\s+2[\.\-:\s]+md&a"]),
            ("risk_factors", [r"(?im)^\s*item\s+1A[\.\-:\s]+risk\s+factors\b"]),
        ]
        slice_items(patterns, text)

    elif form == "8-K":
        # extract item-based segments
        markers = []
        for m in re.finditer(r"(?im)^\s*item\s+([0-9]{1,2}\.[0-9]{2})\b", text):
            item_no = m.group(1)
            markers.append((m.start(), f"item_{item_no.replace('.', '_')}"))
        markers = sorted(markers, key=lambda x: x[0])
        for i, (start, name) in enumerate(markers):
            end = markers[i + 1][0] if i + 1 < len(markers) else len(text)
            chunk = text[start:end].strip()
            if len(chunk) >= 200:
                sections.append({
                    "section_name": name,
                    "section_rank": i + 1,
                    "source_mode": "regex_item",
                    "text": chunk,
                })

    elif form == "DEF 14A":
        heading_patterns = [
            ("executive_compensation", [r"(?im)^\s*executive\s+compensation\b", r"(?im)^\s*compensation\s+discussion"]),
            ("corporate_governance", [r"(?im)^\s*corporate\s+governance\b", r"(?im)^\s*board\s+of\s+directors\b"]),
            ("security_ownership", [r"(?im)^\s*security\s+ownership\b", r"(?im)^\s*beneficial\s+ownership\b"]),
        ]
        slice_items(heading_patterns, text)

    # fallback if no sections found: split by paragraphs and keep the first large chunk
    if not sections:
        paras = [p.strip() for p in re.split(r"\n{2,}", text) if len(p.strip()) >= 250]
        if paras:
            joined = "\n\n".join(paras[:20])
            sections.append({
                "section_name": "full_document_fallback",
                "section_rank": 1,
                "source_mode": "fallback",
                "text": joined,
            })

    return sections


def merge_csv_parts(glob_pattern: str, out_path: Path, header: list[str]) -> int:
    parts = sorted(PARTS_DIR.glob(glob_pattern))
    if not parts:
        return 0
    tmp = out_path.with_suffix(out_path.suffix + ".part")
    count = 0
    with tmp.open("w", newline="", encoding="utf-8") as out_f:
        writer = csv.writer(out_f)
        writer.writerow(header)
        for i, p in enumerate(parts):
            with p.open("r", newline="", encoding="utf-8") as in_f:
                reader = csv.reader(in_f)
                # skip header
                try:
                    next(reader)
                except StopIteration:
                    continue
                for row in reader:
                    writer.writerow(row)
                    count += 1
    tmp.replace(out_path)
    return count


def build_inventory_rows(input_root: Path) -> list[list[Any]]:
    rows = []
    for p in sorted(input_root.rglob("*")):
        if not p.is_file():
            continue
        if p.suffix.lower() not in {".txt", ".html", ".htm"}:
            continue
        rel = str(p.relative_to(input_root)).replace("\\", "/")
        year, form_folder = derive_year_form(rel)
        if form_folder not in ALLOWED_FORM_FOLDERS:
            continue
        rows.append([
            rel,
            str(p),
            year,
            form_folder,
            p.stat().st_size,
            p.name,
        ])
    return rows


# ============================================================
# MODULE A — INVENTORY
# ============================================================

def run_inventory(args: argparse.Namespace) -> None:
    start = now_ts()
    rows = build_inventory_rows(Path(args.input_dir))
    write_csv(INVENTORY_CSV, INVENTORY_HEADER, rows)
    elapsed = now_ts() - start
    print(f"[inventory] rows={len(rows):,} | output={INVENTORY_CSV} | elapsed={fmt_elapsed(elapsed)}", flush=True)


# ============================================================
# MODULE B — CLEAN / PARSE / FILTER / FULL CORPUS
# ============================================================

def process_clean_shard(
    shard_id: int,
    files: list[str],
    input_root_str: str,
    universe_ciks: list[str],
    min_free_bytes: int,
) -> dict[str, Any]:
    input_root = Path(input_root_str)
    universe = set(universe_ciks)

    meta_tmp = PARTS_DIR / f"docs_meta_part_{shard_id:05d}.csv.part"
    text_tmp = PARTS_DIR / f"docs_text_part_{shard_id:05d}.jsonl.gz.part"
    err_tmp = PARTS_DIR / f"docs_errors_part_{shard_id:05d}.csv.part"

    meta_final = PARTS_DIR / f"docs_meta_part_{shard_id:05d}.csv"
    text_final = PARTS_DIR / f"docs_text_part_{shard_id:05d}.jsonl.gz"
    err_final = PARTS_DIR / f"docs_errors_part_{shard_id:05d}.csv"

    # skip if already done
    if meta_final.exists() and text_final.exists() and done_marker(meta_final).exists():
        return {
            "shard_id": shard_id,
            "input_files": 0,
            "ok_files": 0,
            "failed_files": 0,
            "text_records": 0,
            "reused": True,
        }

    ok_files = 0
    failed_files = 0
    text_records = 0

    with meta_tmp.open("w", newline="", encoding="utf-8") as meta_f, \
         gzip.open(text_tmp, "wb", compresslevel=COMPRESSLEVEL) as text_f, \
         err_tmp.open("w", newline="", encoding="utf-8") as err_f:

        meta_w = csv.writer(meta_f)
        err_w = csv.writer(err_f)
        meta_w.writerow(DOC_META_HEADER)
        err_w.writerow(ERRORS_HEADER)

        for idx, file_str in enumerate(files, start=1):
            if idx % 64 == 0:
                if free_bytes(PARTS_DIR) < min_free_bytes:
                    raise RuntimeError("LowDiskSpaceError: free disk dropped below reserve during clean stage")

            path = Path(file_str)
            rel = str(path.relative_to(input_root)).replace("\\", "/")

            try:
                raw = path.read_bytes()
                meta, clean_text = parse_raw_filing(raw, rel, universe)

                # Only keep allowed form canonical names and target universe
                canonical_form = (meta["form_type"] or "").upper().replace("_", " ")
                if canonical_form not in ALLOWED_FORMS:
                    # If SEC header says something odd but folder is target, fall back to folder form.
                    folder_form = folder_form_to_canonical(meta["folder_form"]).upper()
                    if folder_form in ALLOWED_FORMS:
                        meta["form_type"] = folder_form
                    else:
                        meta["status"] = "skipped_non_target_form"

                meta["part_id"] = f"{shard_id:05d}"

                meta_w.writerow([
                    meta["doc_id"], meta["rel_path"], meta["part_id"], meta["year"],
                    meta["form_type"], meta["folder_form"], meta["cik"], meta["entity_name"],
                    meta["filing_date"], meta["period_of_report"], meta["acceptance_datetime"],
                    meta["accession"], meta["primary_doc_type"], meta["primary_doc_filename"],
                    meta["is_company_in_universe"], meta["clean_text_chars"], meta["clean_text_words"],
                    meta["section_hint_count"], meta["status"], meta["error_message"],
                ])

                if meta["is_company_in_universe"] == 1 and meta["status"] == "ok" and clean_text:
                    rec = {
                        "doc_id": meta["doc_id"],
                        "rel_path": meta["rel_path"],
                        "part_id": meta["part_id"],
                        "year": meta["year"],
                        "form_type": meta["form_type"],
                        "cik": meta["cik"],
                        "entity_name": meta["entity_name"],
                        "filing_date": meta["filing_date"],
                        "period_of_report": meta["period_of_report"],
                        "acceptance_datetime": meta["acceptance_datetime"],
                        "accession": meta["accession"],
                        "primary_doc_type": meta["primary_doc_type"],
                        "primary_doc_filename": meta["primary_doc_filename"],
                        "clean_text": clean_text,
                    }
                    text_f.write(json_dumps(rec) + b"\n")
                    text_records += 1
                    ok_files += 1
                else:
                    ok_files += 1

            except Exception as exc:
                failed_files += 1
                doc_id = hash_id(rel)
                err_w.writerow([
                    "clean",
                    rel,
                    doc_id,
                    type(exc).__name__,
                    str(exc),
                ])
                meta_w.writerow([
                    doc_id, rel, f"{shard_id:05d}", "", "", "", "", "", "", "", "", "", "", "",
                    0, 0, 0, 0, "error", str(exc),
                ])

    meta_tmp.replace(meta_final)
    text_tmp.replace(text_final)
    err_tmp.replace(err_final)
    done_marker(meta_final).write_text("done", encoding="utf-8")

    return {
        "shard_id": shard_id,
        "input_files": len(files),
        "ok_files": ok_files,
        "failed_files": failed_files,
        "text_records": text_records,
        "reused": False,
    }


def run_clean(args: argparse.Namespace) -> None:
    start = now_ts()
    input_root = Path(args.input_dir)
    workers = choose_worker_count(args.workers)
    min_free_bytes = int(args.min_free_gb * (1024 ** 3))

    universe = load_cik_union([Path(args.cik_map), Path(args.issuer_tickers)])
    inv_rows = []
    if INVENTORY_CSV.exists():
        with INVENTORY_CSV.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            inv_rows = [row for row in reader]
    else:
        rows = build_inventory_rows(input_root)
        write_csv(INVENTORY_CSV, INVENTORY_HEADER, rows)
        inv_rows = [
            dict(zip(INVENTORY_HEADER, map(str, row)))
            for row in rows
        ]

    all_files = [Path(r["abs_path"]) for r in inv_rows]

    if args.resume:
        processed = load_processed_relpaths("docs_meta_part_*.csv", "rel_path")
        files = [p for p in all_files if str(p.relative_to(input_root)).replace("\\", "/") not in processed]
    else:
        files = all_files

    if not files:
        print("[clean] nothing to do", flush=True)
        return

    shard_count = max(1, min(len(files), workers * max(1, args.shard_multiplier)))
    shards = greedy_partition(files, shard_count)

    print(f"[clean] files={len(files):,} workers={workers} shards={len(shards)} free_disk={human_bytes(free_bytes(OUT_ROOT))}", flush=True)

    total_done = 0
    total_ok = 0
    total_fail = 0
    total_records = 0

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = []
        for i, shard_files in enumerate(shards, start=1):
            futures.append(ex.submit(
                process_clean_shard,
                i,
                [str(p) for p in shard_files],
                str(input_root),
                list(universe),
                min_free_bytes,
            ))

        completed = 0
        for fut in as_completed(futures):
            res = fut.result()
            completed += 1
            total_done += res["input_files"]
            total_ok += res["ok_files"]
            total_fail += res["failed_files"]
            total_records += res["text_records"]
            elapsed = now_ts() - start
            rate = total_done / max(elapsed, 1e-9)
            print(
                f"[clean] shards {completed}/{len(shards)} | files {total_done:,}/{len(files):,} | "
                f"ok {total_ok:,} | fail {total_fail:,} | docs {total_records:,} | rate {rate:,.1f} files/s",
                flush=True
            )

    # Merge metadata and errors; text stays partitioned
    merged_meta = merge_csv_parts("docs_meta_part_*.csv", FULL_DOCS_MANIFEST, DOC_META_HEADER)
    merged_err = merge_csv_parts("docs_errors_part_*.csv", ERRORS_CSV, ERRORS_HEADER)

    elapsed = now_ts() - start
    print(
        f"[clean] done | manifest_rows={merged_meta:,} | errors={merged_err:,} | "
        f"text_parts={len(list(PARTS_DIR.glob('docs_text_part_*.jsonl.gz'))):,} | elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# MODULE C — SECTIONS
# ============================================================


def iter_doc_text_records() -> Iterable[dict[str, Any]]:
    for p in sorted(PARTS_DIR.glob("docs_text_part_*.jsonl.gz")):
        with gzip.open(p, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                yield json.loads(line)


def iter_doc_text_part_paths() -> list[Path]:
    return sorted(PARTS_DIR.glob("docs_text_part_*.jsonl.gz"))




def process_section_part(
    shard_id: int,
    text_part_path_str: str,
    min_free_bytes: int,
) -> dict[str, Any]:
    text_part_path = Path(text_part_path_str)

    meta_tmp = PARTS_DIR / f"sections_meta_part_{shard_id:05d}.csv.part"
    text_tmp = PARTS_DIR / f"sections_text_part_{shard_id:05d}.jsonl.gz.part"
    err_tmp = PARTS_DIR / f"sections_errors_part_{shard_id:05d}.csv.part"

    meta_final = PARTS_DIR / f"sections_meta_part_{shard_id:05d}.csv"
    text_final = PARTS_DIR / f"sections_text_part_{shard_id:05d}.jsonl.gz"
    err_final = PARTS_DIR / f"sections_errors_part_{shard_id:05d}.csv"

    if meta_final.exists() and text_final.exists() and done_marker(meta_final).exists():
        return {
            "shard_id": shard_id,
            "input_docs": 0,
            "section_rows": 0,
            "failed_docs": 0,
            "reused": True,
        }

    input_docs = 0
    section_rows = 0
    failed_docs = 0

    with meta_tmp.open("w", newline="", encoding="utf-8") as meta_f,          gzip.open(text_tmp, "wb", compresslevel=COMPRESSLEVEL) as text_f,          err_tmp.open("w", newline="", encoding="utf-8") as err_f,          gzip.open(text_part_path, "rb") as in_f:

        meta_w = csv.writer(meta_f)
        err_w = csv.writer(err_f)
        meta_w.writerow(SECTION_META_HEADER)
        err_w.writerow(ERRORS_HEADER)

        for idx, line in enumerate(in_f, start=1):
            if not line.strip():
                continue
            if idx % 64 == 0 and free_bytes(PARTS_DIR) < min_free_bytes:
                raise RuntimeError("LowDiskSpaceError: free disk dropped below reserve during sections stage")

            rec = json.loads(line)
            input_docs += 1
            try:
                doc_meta = {
                    "doc_id": rec["doc_id"],
                    "part_id": rec["part_id"],
                    "year": rec["year"],
                    "form_type": rec["form_type"],
                    "cik": rec["cik"],
                    "filing_date": rec["filing_date"],
                    "accession": rec["accession"],
                }
                sections = extract_sections_for_doc(doc_meta, rec["clean_text"])

                for s in sections:
                    sid = hash_id(rec["doc_id"], s["section_name"], str(s["section_rank"]))
                    text_chars = len(s["text"])
                    text_words = count_words(s["text"])
                    meta_w.writerow([
                        sid,
                        rec["doc_id"],
                        f"{shard_id:05d}",
                        rec["year"],
                        rec["form_type"],
                        rec["cik"],
                        rec["filing_date"],
                        rec["accession"],
                        s["section_name"],
                        s["section_rank"],
                        s["source_mode"],
                        text_chars,
                        text_words,
                    ])
                    text_f.write(json_dumps({
                        "section_id": sid,
                        "doc_id": rec["doc_id"],
                        "part_id": f"{shard_id:05d}",
                        "year": rec["year"],
                        "form_type": rec["form_type"],
                        "cik": rec["cik"],
                        "filing_date": rec["filing_date"],
                        "accession": rec["accession"],
                        "section_name": s["section_name"],
                        "section_rank": s["section_rank"],
                        "source_mode": s["source_mode"],
                        "text": s["text"],
                    }) + b"\n")
                    section_rows += 1

            except Exception as exc:
                failed_docs += 1
                err_w.writerow([
                    "sections",
                    rec.get("rel_path", ""),
                    rec.get("doc_id", ""),
                    type(exc).__name__,
                    str(exc),
                ])

    meta_tmp.replace(meta_final)
    text_tmp.replace(text_final)
    err_tmp.replace(err_final)
    done_marker(meta_final).write_text("done", encoding="utf-8")

    return {
        "shard_id": shard_id,
        "input_docs": input_docs,
        "section_rows": section_rows,
        "failed_docs": failed_docs,
        "reused": False,
    }




def run_sections(args: argparse.Namespace) -> None:
    start = now_ts()
    workers = choose_worker_count(args.workers)
    min_free_bytes = int(args.min_free_gb * (1024 ** 3))

    part_paths = iter_doc_text_part_paths()
    if args.resume:
        pending = []
        for i, p in enumerate(part_paths, start=1):
            meta_final = PARTS_DIR / f"sections_meta_part_{i:05d}.csv"
            text_final = PARTS_DIR / f"sections_text_part_{i:05d}.jsonl.gz"
            if meta_final.exists() and text_final.exists() and done_marker(meta_final).exists():
                continue
            pending.append((i, p))
    else:
        pending = [(i, p) for i, p in enumerate(part_paths, start=1)]

    if not pending:
        print("[sections] nothing to do", flush=True)
        return

    total_docs = 0
    total_sections = 0
    total_fail = 0

    print(f"[sections] parts={len(pending):,} workers={workers}", flush=True)

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(process_section_part, shard_id, str(p), min_free_bytes) for shard_id, p in pending]

        completed = 0
        for fut in as_completed(futures):
            res = fut.result()
            completed += 1
            total_docs += res["input_docs"]
            total_sections += res["section_rows"]
            total_fail += res["failed_docs"]
            elapsed = now_ts() - start
            rate = total_docs / max(elapsed, 1e-9)
            print(
                f"[sections] parts {completed}/{len(pending)} | docs {total_docs:,} | "
                f"sections {total_sections:,} | fail_docs {total_fail:,} | rate {rate:,.1f} docs/s",
                flush=True
            )

    merged_meta = merge_csv_parts("sections_meta_part_*.csv", FULL_SECTIONS_MANIFEST, SECTION_META_HEADER)
    merge_csv_parts("sections_errors_part_*.csv", ERRORS_CSV, ERRORS_HEADER)

    elapsed = now_ts() - start
    print(
        f"[sections] done | section_rows={merged_meta:,} | text_parts={len(list(PARTS_DIR.glob('sections_text_part_*.jsonl.gz'))):,} "
        f"| elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# MODULE D — QUALITY / COVERAGE REPORTS
# ============================================================

def read_csv_dicts(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))



def run_quality(args: argparse.Namespace) -> None:
    start = now_ts()

    by_year = Counter()
    by_year_form = Counter()
    by_year_company = Counter()
    by_section = Counter()
    by_doc_section_presence = Counter()
    duplicate_accessions = Counter()
    length_stats = defaultdict(list)
    kept_doc_ids = set()
    docs_total = 0
    docs_kept_count = 0

    if not FULL_DOCS_MANIFEST.exists():
        raise RuntimeError("No full docs manifest rows found. Run clean first.")

    with FULL_DOCS_MANIFEST.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for d in reader:
            docs_total += 1
            if d.get("status") != "ok" or d.get("is_company_in_universe") != "1":
                continue
            docs_kept_count += 1
            kept_doc_ids.add(d["doc_id"])
            year = d["year"]
            form = d["form_type"]
            cik = d["cik"]
            by_year[year] += 1
            by_year_form[(year, form)] += 1
            by_year_company[(year, cik)] += 1
            if d.get("accession"):
                duplicate_accessions[(d["cik"], d["accession"])] += 1
            try:
                length_stats[form].append(int(d["clean_text_words"] or 0))
            except Exception:
                pass

    if not kept_doc_ids:
        raise RuntimeError("No full docs manifest rows found. Run clean first.")

    sections_total = 0
    sections_by_doc = defaultdict(set)
    if FULL_SECTIONS_MANIFEST.exists():
        with FULL_SECTIONS_MANIFEST.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for s in reader:
                if s.get("doc_id") not in kept_doc_ids:
                    continue
                sections_total += 1
                did = s["doc_id"]
                sec_name = s["section_name"]
                sections_by_doc[did].add(sec_name)
                by_section[(s["year"], s["form_type"], sec_name)] += 1

    for did, secs in sections_by_doc.items():
        by_doc_section_presence[len(secs)] += 1

    write_csv(
        YEAR_COVERAGE_CSV,
        ["year", "doc_count"],
        [[y, c] for y, c in sorted(by_year.items())]
    )
    write_csv(
        YEAR_FORM_COVERAGE_CSV,
        ["year", "form_type", "doc_count"],
        [[y, f, c] for (y, f), c in sorted(by_year_form.items())]
    )
    write_csv(
        YEAR_COMPANY_COVERAGE_CSV,
        ["year", "cik", "doc_count"],
        [[y, cik, c] for (y, cik), c in sorted(by_year_company.items())]
    )
    write_csv(
        SECTION_COVERAGE_CSV,
        ["year", "form_type", "section_name", "section_count"],
        [[y, f, s, c] for (y, f, s), c in sorted(by_section.items())]
    )

    summary = {
        "docs_total": docs_total,
        "docs_kept_full_filtered": docs_kept_count,
        "sections_total": sections_total,
        "years_covered": sorted(by_year.keys()),
        "duplicate_accession_pairs_gt1": sum(1 for _, v in duplicate_accessions.items() if v > 1),
        "forms": {
            form: {
                "docs": sum(1 for _ in length_stats[form]),
                "avg_words": (sum(length_stats[form]) / len(length_stats[form])) if length_stats[form] else 0.0,
                "median_words_approx": sorted(length_stats[form])[len(length_stats[form]) // 2] if length_stats[form] else 0,
            }
            for form in FORM_ORDER
        },
        "year_counts": dict(sorted(by_year.items())),
        "section_presence_by_num_sections_per_doc": dict(sorted(by_doc_section_presence.items())),
    }

    with QUALITY_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    elapsed = now_ts() - start
    print(f"[quality] done | docs={docs_kept_count:,} | sections={sections_total:,} | elapsed={fmt_elapsed(elapsed)}", flush=True)


# ============================================================
# MODULE E — FULL + BALANCED DATASETS
# ============================================================

def weighted_form_allocation(target_n: int, available_by_form: dict[str, int], weights: dict[str, float]) -> dict[str, int]:
    quotas = {f: 0 for f in FORM_ORDER}
    remaining = target_n

    # initial weighted allocation
    for f in FORM_ORDER:
        want = int(round(target_n * weights.get(f, 0.0)))
        take = min(want, available_by_form.get(f, 0))
        quotas[f] = take
        remaining -= take

    if remaining <= 0:
        return quotas

    # fill remainder from forms with spare capacity, prioritized by weight then availability
    forms = sorted(FORM_ORDER, key=lambda x: (weights.get(x, 0.0), available_by_form.get(x, 0)), reverse=True)
    while remaining > 0:
        progressed = False
        for f in forms:
            if quotas[f] < available_by_form.get(f, 0):
                quotas[f] += 1
                remaining -= 1
                progressed = True
                if remaining <= 0:
                    break
        if not progressed:
            break
    return quotas


def run_datasets(args: argparse.Namespace) -> None:
    start = now_ts()
    rng = random.Random(args.seed)

    docs = read_csv_dicts(FULL_DOCS_MANIFEST)
    sections = read_csv_dicts(FULL_SECTIONS_MANIFEST)

    docs_kept = [d for d in docs if d.get("status") == "ok" and d.get("is_company_in_universe") == "1"]
    if not docs_kept:
        raise RuntimeError("No full docs manifest rows found. Run clean first.")

    # full dataset manifests (already filtered by universe)
    write_csv(FULL_DOCS_MANIFEST, DOC_META_HEADER, [
        [d.get(col, "") for col in DOC_META_HEADER] for d in docs_kept
    ])

    sections_kept = [s for s in sections if s.get("doc_id") in {d["doc_id"] for d in docs_kept}]
    write_csv(FULL_SECTIONS_MANIFEST, SECTION_META_HEADER, [
        [s.get(col, "") for col in SECTION_META_HEADER] for s in sections_kept
    ])

    by_year = Counter(d["year"] for d in docs_kept if d.get("year"))
    usable_years = [y for y, c in sorted(by_year.items()) if c >= args.min_docs_per_year]
    if not usable_years:
        raise RuntimeError("No usable years satisfy min_docs_per_year; lower the threshold or inspect coverage.")

    weakest_usable_year = min(usable_years, key=lambda y: by_year[y])
    target_per_year = by_year[weakest_usable_year]

    # optional override
    if args.target_per_year > 0:
        target_per_year = min(target_per_year, args.target_per_year)

    weights = DEFAULT_FORM_WEIGHTS.copy()
    if args.form_weights:
        for kv in args.form_weights.split(","):
            if "=" not in kv:
                continue
            k, v = kv.split("=", 1)
            k = k.strip().upper().replace("_", " ")
            if k in weights:
                weights[k] = float(v)

    # normalize weights
    total_w = sum(weights.values()) or 1.0
    weights = {k: v / total_w for k, v in weights.items()}

    docs_by_year_form: dict[str, dict[str, list[dict[str, str]]]] = defaultdict(lambda: defaultdict(list))
    for d in docs_kept:
        if d["year"] in usable_years:
            docs_by_year_form[d["year"]][d["form_type"]].append(d)

    sampled_doc_ids: set[str] = set()
    balanced_rows: list[dict[str, str]] = []
    sampling_audit: list[dict[str, Any]] = []

    for year in sorted(usable_years):
        available_by_form = {f: len(docs_by_year_form[year].get(f, [])) for f in FORM_ORDER}
        quotas = weighted_form_allocation(target_per_year, available_by_form, weights)

        for form in FORM_ORDER:
            candidates = docs_by_year_form[year].get(form, [])
            # optional company cap
            if args.max_docs_per_cik_per_year_form > 0:
                by_cik = defaultdict(list)
                for d in candidates:
                    by_cik[d["cik"]].append(d)
                capped = []
                for cik, rows in by_cik.items():
                    rows = rows[:]
                    rng.shuffle(rows)
                    capped.extend(rows[:args.max_docs_per_cik_per_year_form])
                candidates = capped

            rng.shuffle(candidates)
            chosen = candidates[:quotas.get(form, 0)]
            for d in chosen:
                sampled_doc_ids.add(d["doc_id"])
                balanced_rows.append(d)

        sampling_audit.append({
            "year": year,
            "available_total": by_year[year],
            "target_total": target_per_year,
            "available_by_form": available_by_form,
            "quotas": quotas,
        })

    balanced_sections = [s for s in sections_kept if s["doc_id"] in sampled_doc_ids]

    write_csv(BALANCED_DOCS_MANIFEST, DOC_META_HEADER, [
        [d.get(col, "") for col in DOC_META_HEADER] for d in balanced_rows
    ])
    write_csv(BALANCED_SECTIONS_MANIFEST, SECTION_META_HEADER, [
        [s.get(col, "") for col in SECTION_META_HEADER] for s in balanced_sections
    ])

    with BALANCE_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump({
            "seed": args.seed,
            "min_docs_per_year": args.min_docs_per_year,
            "usable_years": usable_years,
            "weakest_usable_year": weakest_usable_year,
            "target_per_year": target_per_year,
            "form_weights": weights,
            "balanced_doc_count": len(balanced_rows),
            "balanced_section_count": len(balanced_sections),
            "sampling_audit": sampling_audit,
        }, f, indent=2)

    elapsed = now_ts() - start
    print(
        f"[datasets] done | usable_years={len(usable_years):,} | target_per_year={target_per_year:,} | "
        f"balanced_docs={len(balanced_rows):,} | balanced_sections={len(balanced_sections):,} | elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# MODULE F — FINBERT-READY CHUNKS
# ============================================================

SECTION_PRIORITY = {
    "risk_factors": 1,
    "mda": 2,
    "business": 3,
    "financial_statements": 4,
    "executive_compensation": 5,
    "corporate_governance": 6,
    "security_ownership": 7,
    "full_document_fallback": 99,
}



def iter_section_text_records() -> Iterable[dict[str, Any]]:
    for p in sorted(PARTS_DIR.glob("sections_text_part_*.jsonl.gz")):
        with gzip.open(p, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                yield json.loads(line)


def chunk_words(text: str, chunk_size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    i = 0
    step = max(1, chunk_size - overlap)
    while i < len(words):
        chunk = words[i:i + chunk_size]
        if len(chunk) >= max(50, chunk_size // 3):
            chunks.append(" ".join(chunk))
        i += step
    return chunks



def load_doc_text_map() -> dict[str, dict[str, Any]]:
    m = {}
    for p in sorted(PARTS_DIR.glob("docs_text_part_*.jsonl.gz")):
        with gzip.open(p, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)
                m[rec["doc_id"]] = rec
    return m


def chunk_words(text: str, chunk_size: int, overlap: int) -> list[str]:
    words = text.split()
    if not words:
        return []
    chunks = []
    i = 0
    step = max(1, chunk_size - overlap)
    while i < len(words):
        chunk = words[i:i + chunk_size]
        if len(chunk) >= max(50, chunk_size // 3):
            chunks.append(" ".join(chunk))
        i += step
    return chunks



def run_finbert(args: argparse.Namespace) -> None:
    start = now_ts()
    full_docs = read_csv_dicts(FULL_DOCS_MANIFEST)
    balanced_docs = read_csv_dicts(BALANCED_DOCS_MANIFEST)

    full_doc_ids = {d["doc_id"] for d in full_docs}
    balanced_doc_ids = {d["doc_id"] for d in balanced_docs}

    if not full_doc_ids:
        raise RuntimeError("No full docs manifest found for finbert stage.")
    if not FULL_SECTIONS_MANIFEST.exists():
        raise RuntimeError("No sections manifest found for finbert stage. Run sections first.")

    header = [
        "chunk_id",
        "doc_id",
        "year",
        "form_type",
        "cik",
        "filing_date",
        "accession",
        "source_name",
        "chunk_index",
        "word_count",
        "text",
    ]

    all_tmp = FINBERT_CHUNKS_ALL_CSV.with_suffix(FINBERT_CHUNKS_ALL_CSV.suffix + ".part")
    bal_tmp = FINBERT_CHUNKS_BALANCED_CSV.with_suffix(FINBERT_CHUNKS_BALANCED_CSV.suffix + ".part")

    all_count = 0
    bal_count = 0

    with all_tmp.open("w", newline="", encoding="utf-8") as all_f,          bal_tmp.open("w", newline="", encoding="utf-8") as bal_f:
        all_w = csv.writer(all_f)
        bal_w = csv.writer(bal_f)
        all_w.writerow(header)
        bal_w.writerow(header)

        current_doc_id = None
        current_sections = []

        def flush_doc_sections(doc_id: str, doc_sections: list[dict[str, Any]]) -> tuple[int, int]:
            if not doc_id or not doc_sections:
                return 0, 0

            doc_sections_sorted = sorted(
                doc_sections,
                key=lambda x: (SECTION_PRIORITY.get(x["section_name"], 50), int(x.get("section_rank", 999)))
            )
            selected = []
            for s in doc_sections_sorted[:args.max_sections_per_doc]:
                if len(s["text"]) >= args.min_section_chars:
                    selected.append((s["section_name"], s["text"], s))
            if not selected:
                s = doc_sections_sorted[0]
                selected = [(s["section_name"], s["text"], s)]

            local_all = 0
            local_bal = 0
            chunk_idx = 0
            for source_name, source_text, meta in selected:
                for ch in chunk_words(source_text, args.chunk_words, args.overlap_words):
                    chunk_idx += 1
                    row = [
                        hash_id(doc_id, source_name, str(chunk_idx)),
                        doc_id,
                        meta["year"],
                        meta["form_type"],
                        meta["cik"],
                        meta["filing_date"],
                        meta["accession"],
                        source_name,
                        chunk_idx,
                        count_words(ch),
                        ch,
                    ]
                    all_w.writerow(row)
                    local_all += 1
                    if doc_id in balanced_doc_ids:
                        bal_w.writerow(row)
                        local_bal += 1
            return local_all, local_bal

        for sec in iter_section_text_records():
            did = sec["doc_id"]
            if did not in full_doc_ids:
                continue
            if current_doc_id is None:
                current_doc_id = did
            if did != current_doc_id:
                a, b = flush_doc_sections(current_doc_id, current_sections)
                all_count += a
                bal_count += b
                current_doc_id = did
                current_sections = [sec]
            else:
                current_sections.append(sec)

        if current_sections:
            a, b = flush_doc_sections(current_doc_id, current_sections)
            all_count += a
            bal_count += b

    all_tmp.replace(FINBERT_CHUNKS_ALL_CSV)
    bal_tmp.replace(FINBERT_CHUNKS_BALANCED_CSV)

    with FINBERT_SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump({
            "chunk_words": args.chunk_words,
            "overlap_words": args.overlap_words,
            "max_sections_per_doc": args.max_sections_per_doc,
            "min_section_chars": args.min_section_chars,
            "all_chunk_count": all_count,
            "balanced_chunk_count": bal_count,
        }, f, indent=2)

    elapsed = now_ts() - start
    print(
        f"[finbert] done | all_chunks={all_count:,} | balanced_chunks={bal_count:,} | elapsed={fmt_elapsed(elapsed)}",
        flush=True
    )


# ============================================================
# ALL
# ============================================================

def run_all(args: argparse.Namespace) -> None:
    if not INVENTORY_CSV.exists() or args.force_inventory:
        run_inventory(args)
    run_clean(args)
    run_sections(args)
    run_quality(args)
    run_datasets(args)
    run_finbert(args)


# ============================================================
# CLI
# ============================================================

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Comprehensive HDD-aware SEC filings text cleaning + engineering pipeline."
    )
    sub = p.add_subparsers(dest="command", required=True)

    def common(sp):
        sp.add_argument("--input-dir", type=str, default=str(RAW_FILINGS_DIR))
        sp.add_argument("--cik-map", type=str, default=str(DEFAULT_CIK_MAP))
        sp.add_argument("--issuer-tickers", type=str, default=str(DEFAULT_ISSUER_TICKERS))
        sp.add_argument("--workers", type=int, default=0)
        sp.add_argument("--shard-multiplier", type=int, default=2)
        sp.add_argument("--resume", action="store_true")
        sp.add_argument("--overwrite", action="store_true")
        sp.add_argument("--min-free-gb", type=float, default=8.0)

    s = sub.add_parser("inventory")
    common(s)

    s = sub.add_parser("clean")
    common(s)

    s = sub.add_parser("sections")
    common(s)

    s = sub.add_parser("quality")
    common(s)

    s = sub.add_parser("datasets")
    common(s)
    s.add_argument("--seed", type=int, default=42)
    s.add_argument("--min-docs-per-year", type=int, default=1000)
    s.add_argument("--target-per-year", type=int, default=0)
    s.add_argument("--max-docs-per-cik-per-year-form", type=int, default=5)
    s.add_argument("--form-weights", type=str, default="")

    s = sub.add_parser("finbert")
    common(s)
    s.add_argument("--chunk-words", type=int, default=350)
    s.add_argument("--overlap-words", type=int, default=50)
    s.add_argument("--max-sections-per-doc", type=int, default=3)
    s.add_argument("--min-section-chars", type=int, default=500)

    s = sub.add_parser("all")
    common(s)
    s.add_argument("--force-inventory", action="store_true")
    s.add_argument("--seed", type=int, default=42)
    s.add_argument("--min-docs-per-year", type=int, default=1000)
    s.add_argument("--target-per-year", type=int, default=0)
    s.add_argument("--max-docs-per-cik-per-year-form", type=int, default=5)
    s.add_argument("--form-weights", type=str, default="")
    s.add_argument("--chunk-words", type=int, default=350)
    s.add_argument("--overlap-words", type=int, default=50)
    s.add_argument("--max-sections-per-doc", type=int, default=3)
    s.add_argument("--min-section-chars", type=int, default=500)

    return p.parse_args()


def main() -> None:
    args = parse_args()
    ensure_dirs(args.overwrite)

    print(f"Input directory : {args.input_dir}", flush=True)
    print(f"Output root     : {OUT_ROOT}", flush=True)
    print(f"Workers         : {choose_worker_count(args.workers)}", flush=True)
    print(f"Resume mode     : {args.resume}", flush=True)
    print(f"Min free space  : {args.min_free_gb:.2f} GB", flush=True)
    print(f"JSON parser     : {'orjson' if HAS_ORJSON else 'stdlib json'}", flush=True)

    cmd = args.command
    if cmd == "inventory":
        run_inventory(args)
    elif cmd == "clean":
        run_clean(args)
    elif cmd == "sections":
        run_sections(args)
    elif cmd == "quality":
        run_quality(args)
    elif cmd == "datasets":
        run_datasets(args)
    elif cmd == "finbert":
        run_finbert(args)
    elif cmd == "all":
        run_all(args)
    else:
        raise SystemExit(f"Unknown command: {cmd}")


if __name__ == "__main__":
    main()



### data\sec_filings_topup_manifest.py

In [ ]:
#!/usr/bin/env python3
"""
sec_filings_topup_manifest.py

Detect weak years in the already-processed filings text corpus and generate a small
targeted SEC download manifest that tops those years up to the current balanced target.

This script does NOT hallucinate missing data. It only creates a targeted download list.
That is the only honest way to "bring weak years up to the same level" without corrupting
the text corpus with synthetic oversampling.
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import random
from collections import defaultdict, Counter, deque
from pathlib import Path
from typing import Any

dataPath = Path(os.getenv("dataPathGlobal", "data"))

CLEANED_DIR = dataPath / "sec_edgar" / "processed" / "cleaned"
FILINGS_TEXT_DIR = dataPath / "sec_edgar" / "processed" / "filings_text"
MANIFEST_DIR = dataPath / "sec_edgar" / "processed" / "manifests"

DEFAULT_CIK_MAP = CLEANED_DIR / "cik_ticker_map_cleaned.csv"
DEFAULT_ISSUER_TICKERS = CLEANED_DIR / "issuer_master_onlyTickers.csv"

DEFAULT_YEAR_COVERAGE = FILINGS_TEXT_DIR / "filings_year_coverage.csv"
DEFAULT_BALANCE_SUMMARY = FILINGS_TEXT_DIR / "filings_balance_summary.json"
DEFAULT_FULL_DOCS = FILINGS_TEXT_DIR / "filings_filtered_full.csv"

DEFAULT_SOURCE_MANIFEST_COMPANYSCOPE = MANIFEST_DIR / "filings_manifest_core_companyscope_2000_2024.csv"
DEFAULT_SOURCE_MANIFEST_MERGED = MANIFEST_DIR / "filings_manifest_2000_2024_merged.csv"

DEFAULT_OUTPUT_MANIFEST = MANIFEST_DIR / "filings_manifest_weak_years_topup.csv"
DEFAULT_OUTPUT_SUMMARY = FILINGS_TEXT_DIR / "filings_topup_summary.json"

ALLOWED_FORMS = {"10-K", "10-Q", "8-K", "DEF 14A"}
FORM_ORDER = ["10-K", "10-Q", "8-K", "DEF 14A"]

DEFAULT_FORM_WEIGHTS = {
    "10-K": 0.30,
    "10-Q": 0.30,
    "8-K": 0.30,
    "DEF 14A": 0.10,
}


def normalize_cik(value: Any) -> str:
    s = str(value).strip()
    if not s:
        return ""
    s = s.lstrip("0")
    return s if s else "0"


def detect_cik_column(fieldnames: list[str]) -> str:
    lowered = {c.lower(): c for c in fieldnames}
    for cand in ("cik", "padded_cik", "sec_cik", "issuer_cik"):
        if cand in lowered:
            return lowered[cand]
    raise RuntimeError(f"Could not find a CIK column in {fieldnames}")


def load_cik_union(csv_paths: list[Path]) -> set[str]:
    cik_set: set[str] = set()
    for path in csv_paths:
        if not path.exists():
            raise FileNotFoundError(f"Required CIK source file not found: {path}")
        with path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fieldnames = reader.fieldnames or []
            cik_col = detect_cik_column(fieldnames)
            loaded = 0
            for row in reader:
                cik = normalize_cik(row.get(cik_col, ""))
                if cik:
                    cik_set.add(cik)
                    loaded += 1
            print(f"[cik-union] loaded {loaded:,} rows from {path} | cumulative unique CIKs={len(cik_set):,}", flush=True)
    return cik_set


def parse_form_weights(text: str) -> dict[str, float]:
    weights = DEFAULT_FORM_WEIGHTS.copy()
    if text:
        for kv in text.split(","):
            if "=" not in kv:
                continue
            k, v = kv.split("=", 1)
            k = k.strip().upper().replace("_", " ")
            if k in weights:
                weights[k] = float(v)
    total = sum(weights.values()) or 1.0
    return {k: v / total for k, v in weights.items()}


def load_target_per_year(balance_summary_path: Path, override_target: int) -> int:
    if override_target > 0:
        return override_target
    if not balance_summary_path.exists():
        raise RuntimeError(
            f"Could not find balance summary at {balance_summary_path} and no --target-per-year was given."
        )
    with balance_summary_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    target = int(data.get("target_per_year", 0) or 0)
    if target <= 0:
        raise RuntimeError("target_per_year missing/invalid in balance summary.")
    return target


def load_year_coverage(path: Path) -> dict[str, int]:
    if not path.exists():
        raise FileNotFoundError(path)
    out: dict[str, int] = {}
    with path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            year = (row.get("year") or "").strip()
            count = int(row.get("doc_count") or 0)
            if year:
                out[year] = count
    return out


def load_existing_doc_keys(full_docs_path: Path) -> tuple[set[str], set[str]]:
    relpaths: set[str] = set()
    cik_acc: set[str] = set()
    if not full_docs_path.exists():
        return relpaths, cik_acc

    with full_docs_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            relp = (row.get("rel_path") or "").replace("\\", "/").strip()
            if relp:
                relpaths.add(relp)

            cik = normalize_cik(row.get("cik", ""))
            accession = (row.get("accession") or "").strip()
            if cik and accession:
                cik_acc.add(f"{cik}|{accession}")
    return relpaths, cik_acc


def year_from_date(date_str: str) -> str:
    if not date_str:
        return ""
    return date_str[:4]


def weighted_form_quotas(deficit: int, available_by_form: dict[str, int], weights: dict[str, float]) -> dict[str, int]:
    quotas = {f: 0 for f in FORM_ORDER}
    remaining = deficit

    for f in FORM_ORDER:
        want = int(round(deficit * weights.get(f, 0.0)))
        take = min(want, available_by_form.get(f, 0))
        quotas[f] = take
        remaining -= take

    if remaining <= 0:
        return quotas

    forms = sorted(FORM_ORDER, key=lambda x: (weights.get(x, 0.0), available_by_form.get(x, 0)), reverse=True)
    while remaining > 0:
        progressed = False
        for f in forms:
            if quotas[f] < available_by_form.get(f, 0):
                quotas[f] += 1
                remaining -= 1
                progressed = True
                if remaining <= 0:
                    break
        if not progressed:
            break
    return quotas


def choose_default_manifest() -> Path:
    if DEFAULT_SOURCE_MANIFEST_COMPANYSCOPE.exists():
        return DEFAULT_SOURCE_MANIFEST_COMPANYSCOPE
    return DEFAULT_SOURCE_MANIFEST_MERGED


def manifest_row_allowed(row: dict[str, str], cik_union: set[str], weak_years: set[str], allowed_forms: set[str]) -> bool:
    form = (row.get("form_type") or "").strip().upper()
    cik = normalize_cik(row.get("cik", ""))
    date_filed = (row.get("date_filed") or "").strip()
    year = year_from_date(date_filed)

    if form not in allowed_forms:
        return False
    if year not in weak_years:
        return False
    if cik not in cik_union:
        return False
    return True


def rr_select_diverse(rows: list[dict[str, str]], quota: int, seed: int) -> list[dict[str, str]]:
    if quota <= 0 or not rows:
        return []

    rng = random.Random(seed)
    by_cik: dict[str, list[dict[str, str]]] = defaultdict(list)
    for r in rows:
        by_cik[normalize_cik(r.get("cik", ""))].append(r)

    cik_order = list(by_cik.keys())
    rng.shuffle(cik_order)
    for cik in cik_order:
        rng.shuffle(by_cik[cik])

    queues = deque(cik_order)
    selected: list[dict[str, str]] = []

    while queues and len(selected) < quota:
        cik = queues.popleft()
        bucket = by_cik[cik]
        if bucket:
            selected.append(bucket.pop())
        if bucket:
            queues.append(cik)

    return selected


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Generate a targeted top-up SEC filings manifest for weak years only."
    )
    p.add_argument("--cik-map", type=str, default=str(DEFAULT_CIK_MAP))
    p.add_argument("--issuer-tickers", type=str, default=str(DEFAULT_ISSUER_TICKERS))
    p.add_argument("--year-coverage", type=str, default=str(DEFAULT_YEAR_COVERAGE))
    p.add_argument("--balance-summary", type=str, default=str(DEFAULT_BALANCE_SUMMARY))
    p.add_argument("--full-docs", type=str, default=str(DEFAULT_FULL_DOCS))
    p.add_argument("--source-manifest", type=str, default=str(choose_default_manifest()))
    p.add_argument("--output-manifest", type=str, default=str(DEFAULT_OUTPUT_MANIFEST))
    p.add_argument("--output-summary", type=str, default=str(DEFAULT_OUTPUT_SUMMARY))
    p.add_argument("--target-per-year", type=int, default=0)
    p.add_argument("--max-topup-per-year", type=int, default=0)
    p.add_argument("--form-weights", type=str, default="")
    p.add_argument("--seed", type=int, default=42)
    return p.parse_args()


def main() -> None:
    args = parse_args()

    cik_union = load_cik_union([Path(args.cik_map), Path(args.issuer_tickers)])
    target_per_year = load_target_per_year(Path(args.balance_summary), args.target_per_year)
    year_cov = load_year_coverage(Path(args.year_coverage))
    processed_relpaths, processed_cik_acc = load_existing_doc_keys(Path(args.full_docs))
    weights = parse_form_weights(args.form_weights)

    weak_years = {}
    for year, count in sorted(year_cov.items()):
        if count < target_per_year:
            deficit = target_per_year - count
            if args.max_topup_per_year > 0:
                deficit = min(deficit, args.max_topup_per_year)
            weak_years[year] = deficit

    if not weak_years:
        print("[topup] No weak years detected. Nothing to do.", flush=True)
        return

    print(f"[topup] target_per_year={target_per_year:,}", flush=True)
    print(f"[topup] weak years detected: {weak_years}", flush=True)

    source_manifest = Path(args.source_manifest)
    if not source_manifest.exists():
        raise FileNotFoundError(f"Source manifest not found: {source_manifest}")

    candidates_by_year_form: dict[str, dict[str, list[dict[str, str]]]] = defaultdict(lambda: defaultdict(list))
    scanned = 0
    kept_candidates = 0
    skipped_existing = 0
    skipped_processed_accession = 0

    with source_manifest.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        if not fieldnames:
            raise RuntimeError(f"No header in source manifest: {source_manifest}")

        required = {"cik", "form_type", "date_filed", "output_path", "filing_url"}
        missing = required - set(fieldnames)
        if missing:
            raise RuntimeError(f"Source manifest missing required columns: {sorted(missing)}")

        for row in reader:
            scanned += 1

            if not manifest_row_allowed(row, cik_union, set(weak_years.keys()), ALLOWED_FORMS):
                continue

            output_path = (row.get("output_path") or "").replace("\\", "/").strip()
            if output_path in processed_relpaths:
                skipped_existing += 1
                continue

            cik = normalize_cik(row.get("cik", ""))
            accession = (row.get("accession") or "").strip()
            if cik and accession and f"{cik}|{accession}" in processed_cik_acc:
                skipped_processed_accession += 1
                continue

            year = year_from_date(row.get("date_filed", ""))
            form = (row.get("form_type") or "").strip().upper()

            candidates_by_year_form[year][form].append(row)
            kept_candidates += 1

            if scanned % 500_000 == 0:
                print(
                    f"[topup] scanned={scanned:,} candidate_rows={kept_candidates:,} "
                    f"skip_existing={skipped_existing:,} skip_processed_accession={skipped_processed_accession:,}",
                    flush=True,
                )

    final_rows: list[dict[str, str]] = []
    planning = {}

    for year in sorted(weak_years.keys()):
        deficit = weak_years[year]
        available_by_form = {f: len(candidates_by_year_form[year].get(f, [])) for f in FORM_ORDER}
        quotas = weighted_form_quotas(deficit, available_by_form, weights)

        selected_for_year: list[dict[str, str]] = []
        for i, form in enumerate(FORM_ORDER, start=1):
            candidates = candidates_by_year_form[year].get(form, [])
            selected = rr_select_diverse(candidates, quotas.get(form, 0), args.seed + i + int(year))
            selected_for_year.extend(selected)

        still_need = deficit - len(selected_for_year)
        if still_need > 0:
            leftovers = []
            selected_keys = {
                (
                    normalize_cik(r.get("cik", "")),
                    (r.get("accession") or "").strip(),
                    (r.get("filing_url") or "").strip(),
                )
                for r in selected_for_year
            }
            for form in FORM_ORDER:
                for r in candidates_by_year_form[year].get(form, []):
                    key = (
                        normalize_cik(r.get("cik", "")),
                        (r.get("accession") or "").strip(),
                        (r.get("filing_url") or "").strip(),
                    )
                    if key not in selected_keys:
                        leftovers.append(r)
            extra = rr_select_diverse(leftovers, still_need, args.seed + 999 + int(year))
            selected_for_year.extend(extra)

        final_rows.extend(selected_for_year)

        planning[year] = {
            "current_count": year_cov.get(year, 0),
            "target_per_year": target_per_year,
            "deficit_requested": deficit,
            "available_by_form": available_by_form,
            "quotas": quotas,
            "selected_count": len(selected_for_year),
            "selected_by_form": dict(Counter((r.get("form_type") or "").strip().upper() for r in selected_for_year)),
        }

    output_manifest = Path(args.output_manifest)
    output_manifest.parent.mkdir(parents=True, exist_ok=True)

    with source_manifest.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames or []

    unique_rows = []
    seen_urls = set()
    for row in final_rows:
        url = (row.get("filing_url") or "").strip()
        if not url or url in seen_urls:
            continue
        seen_urls.add(url)
        unique_rows.append(row)

    tmp_manifest = output_manifest.with_suffix(output_manifest.suffix + ".part")
    with tmp_manifest.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in unique_rows:
            writer.writerow(row)
    tmp_manifest.replace(output_manifest)

    summary = {
        "target_per_year": target_per_year,
        "weak_years": weak_years,
        "weights": weights,
        "source_manifest": str(source_manifest),
        "output_manifest": str(output_manifest),
        "scanned_rows": scanned,
        "candidate_rows_kept": kept_candidates,
        "skipped_existing_relpath": skipped_existing,
        "skipped_existing_cik_accession": skipped_processed_accession,
        "final_topup_rows": len(unique_rows),
        "planning": planning,
    }

    output_summary = Path(args.output_summary)
    output_summary.parent.mkdir(parents=True, exist_ok=True)
    with output_summary.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"[topup] wrote targeted manifest: {output_manifest}", flush=True)
    print(f"[topup] rows={len(unique_rows):,}", flush=True)
    print(f"[topup] summary={output_summary}", flush=True)


if __name__ == "__main__":
    main()

#  run:
# nice -n -20 python -u data/sec_filings_topup_manifest.py
# Rerun the text pipeline incrementally:
# python data/sec_filings_text_pipeline_v2.py inventory
# python data/sec_filings_text_pipeline_v2.py clean --resume
# python data/sec_filings_text_pipeline_v2.py sections --resume
# python data/sec_filings_text_pipeline_v2.py quality
# python data/sec_filings_text_pipeline_v2.py datasets
# python data/sec_filings_text_pipeline_v2.py finbert
